# ICT-36 — F-Lens : mode factored-geometry, sous-espaces et additivite

**Navigation** : [<< ICT-35](ICT-35-HumorCausalProbe-Pilot-Python.ipynb) | [>> ICT-36b (Ruzsa)](ICT-36b-RuzsaOnFactorGeometry-Python.ipynb) | [Index](../README.md)

**Module :** ICT-Series (sous-ensemble F-Lens, Epic #15475)
**Niveau :** Recherche
**Type :** Grain DEEP/notebook-python CONTENU
**Duree estimee :** 45 minutes (lecture + execution)
**VRAM :** 0 (CPU uniquement, primitives numpy-only)

> Ce notebook est un **grain F-Lens factored-geometry autonome** : il ne depend pas du contrat de trace v1 (#15476) ni du mode belief-state (#15477). Il reimplemente ses primitives en numpy-only, sur donnees synthetiques, et documente la migration future vers le contrat commun.

## Question scientifique

Comment les facteurs independants d'un processus predictif occupent-ils le residual stream : sous-espaces additifs, superposes, orthogonaux ou entremeles ?

Le mode factored-geometry etudie l'**organisation geometrique** des facteurs ; il se distingue du mode belief-state (acces predictif lineaire, voir grain #15477) et de la SAE/J-Lens (features latentes et lecture fonctionnelle).

## References

- arXiv:2602.02385 (factored representations) — bibliotheque canonique CoursIA.
- #15478 (issue grain) ; Part of #15475 (Epic toolkit multi-instrument) ; Depends on #15476 (contrat de trace v1) ; complements #15477 (F-Lens belief-state).


> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## Sources et substance (reventilation c.1119)

**Papier fondateur** : Shai et al., *Transformers Learn Factored Representations* ([arXiv:2602.02385](https://arxiv.org/abs/2602.02385)). PDF en biblio canonique : `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\2026 - Shai et al - Transformers Learn Factored Representations.pdf`.

**Dépôts de référence** (non copiés, réimplémentation from-scratch conforme au mandat Epic #15475) :

- [Astera-org/factored-reps](https://github.com/Astera-org/factored-reps) : PyTorch/Hydra. `fwh_core.analysis.pca.compute_weighted_pca` (PCA pondérée + seuils `(0.80, 0.90, 0.95, 0.99)`), `fwh_core.analysis.linear_regression` (R²/RMSE/MAE/dist pondérés), `fwh_core.generative_processes.factored_generative_process.FactoredGenerativeProcess`, `structures/` (`independent`, `fully_conditional`, `sequential_conditional`, `conditional_transitions`).
- [Astera-org/simplexity](https://github.com/Astera-org/simplexity) : JAX/Equinox, **portage structurellement isomorphe de `factored-reps`** (mêmes modules `analysis/pca.py`, `analysis/linear_regression.py`, `generative_processes/factored_generative_process.py`, `structures/`). Le présent notebook reproduit l'API numpy-first offline dans `ict/`, conforme à la règle Epic #15475 « cœur analytique numpy-first/offline ».

**Référençage ICt-37 sibling** : ICT-37 (mode belief-state, PR #15662) traite l'autre moitié du papier — la **décodabilité linéaire** du belief state. Le présent ICT-36 traite la **factorisation géométrique** des sous-espaces. Les deux notebooks sont complémentaires et **non interchangeables** : voir la cellule « Distinction belief vs factorisation » plus bas.

**Pas de licence détectée** sur les deux dépôts Astera (cf. preflight Epic #15475) : la substance LIVRÉE est donc une **réimplémentation from-scratch** des méthodes publiées, sans copie de code. Le test `H.1 — Pondération et centrage explicites et testés` honore cette contrainte.


In [1]:
# Parametres du notebook
nb_name = "ICT-36-FLens-FactoredGeometry-Python"
N_SEEDS = 5  # multi-seed : au moins 4 seeds requis (cf .claude/rules/pr-review-discipline.md, critere C)
N_DIM_RESIDUAL = 256  # dimension typique d'un residual stream LLM (proxy)
N_TOKENS = 2048  # nombre de positions capturees (echantillon)
N_FACTORS = 4  # facteurs independants dans le regime factorise
NC_THRESHOLDS = (80, 90, 95, 99)  # seuils de variance cumulee (NC@k)
NOISE_LEVELS = (0.0, 0.1, 0.25, 0.5, 1.0)  # sweep de bruit
RNG_SEEDS = tuple(range(N_SEEDS))  # (0, 1, 2, 3, 4)
print(f"N_SEEDS={N_SEEDS} RNG_SEEDS={RNG_SEEDS}")
print(f"N_DIM_RESIDUAL={N_DIM_RESIDUAL} N_FACTORS={N_FACTORS} N_TOKENS={N_TOKENS}")


N_SEEDS=5 RNG_SEEDS=(0, 1, 2, 3, 4)
N_DIM_RESIDUAL=256 N_FACTORS=4 N_TOKENS=2048


In [2]:
# Racine canonique de la serie : remonte les parents depuis le dossier du
# notebook jusqu'au package ict/, puis en derive les dossiers de donnees.
# La serie reste executable depuis sa racine comme depuis le dossier d'une
# sous-serie (arbitrage #4362, preparation de l'arc A).
import sys
from pathlib import Path

ICT_ROOT = Path.cwd()
while not (ICT_ROOT / "ict" / "__init__.py").exists() and ICT_ROOT != ICT_ROOT.parent:
    ICT_ROOT = ICT_ROOT.parent
assert (ICT_ROOT / "ict" / "__init__.py").exists(), (
    f"package ict/ introuvable en remontant depuis {Path.cwd().name}")
if str(ICT_ROOT) not in sys.path:
    sys.path.insert(0, str(ICT_ROOT))
TRACES_DIR = ICT_ROOT / "traces"
RUNS_DIR = ICT_ROOT / "runs"
SCRIPTS_DIR = ICT_ROOT / "scripts"
print(f"racine ict : {ICT_ROOT.name}")


racine ict : ICT-Series


In [3]:
# Imports : numpy pour le pilotage des primitives (acceptance #15478).
# Depuis l'extraction #15943, les 7 primitives F-Lens vivent dans le paquet ict/
# (le notebook est la lecon, le paquet est l'outil) -- ce notebook les importe,
# il ne les redefinit plus.
import os
import sys

import numpy as np
import json
from pathlib import Path

from ict.factor_geometry import (
    weighted_pca,
    nc_at,
    basis_overlap,
    max_principal_angle,
    make_factor_bases,
    synthesize_activations,
    null_overlap_distribution,
)

print(f"numpy {np.__version__}")
print("Primitives F-Lens importees depuis ict.factor_geometry (extraction #15943)")

numpy 2.5.2
Primitives F-Lens importees depuis ict.factor_geometry (extraction #15943)


## Primitives numpy-only (PCA ponderee, NC@p, angles, overlap)

Depuis l'extraction #15943, elles vivent dans le paquet `ict/factor_geometry.py` (`from ict.factor_geometry import ...`) : ce notebook les importe et les exerce, il ne les redéfinit plus.

Les primitives sont **autonomes** : elles n'utilisent que `numpy`. La motivation est triple :

1. **Reproductibilite** : pas de dependance a la version sklearn, resultats byte-identiques entre machines.
2. **Pondération explicite** : la ponderation par nombre d'occurrences est necessaire quand certains etats du belief sont sur-representes dans le sample ; sklearn PCA standard ne supporte que `sample_weight` au fit, pas au centrage.
3. **Migration vers le contrat de trace v1 (#15476)** : les primitives consomment un dict `{activations, weights}` et rendent un dict structure ; un futur contrat remplacera l'entree par un load NPZ sans changer les primitives.


In [4]:
# weighted_pca, nc_at, basis_overlap, max_principal_angle : importees de
# ict/factor_geometry.py (extraction #15943, corps deplaces a l'identique,
# resultats inchanges par rapport a la livraison initiale #15514).
# Smoke-check d'import : chaque primitive repond sur un cas toy.
_rng = np.random.default_rng(0)
_mu, _comps, _sv, _evr = weighted_pca(_rng.standard_normal((50, 6)))
_nc = nc_at(np.array([0.5, 0.3, 0.15, 0.05]))[95]
_B_toy = np.eye(6)[:, :3]
_ov = float(basis_overlap(_B_toy, _B_toy).max())
_ang = max_principal_angle(_B_toy, _B_toy)
print(f"smoke: NC@95={_nc} overlap_diag={_ov:.1f} angle_self={_ang:.1f} deg")
print("Primitives PCA ponderee + NC@p + basis_overlap + max_principal_angle pretes.")

smoke: NC@95=3 overlap_diag=1.0 angle_self=0.0 deg
Primitives PCA ponderee + NC@p + basis_overlap + max_principal_angle pretes.


## Generation de donnees synthetiques (trois regimes)

Trois regimes controles, pour tester la discrimination des primitives :

1. **Orthogonal** : chaque facteur occupe un sous-espace de dimension 8 dans `R^256`, disjoint. L'additivite dimensionnelle devrait tenir exactement.
2. **Superpose** : les sous-espaces se chevauchent partiellement (overlap ~ 0.4). L'additivite est degradee.
3. **Bruite** : orthogonal + bruit additif gaussien de niveaux croissants. La structure factorisee devrait disparaitre progressivement.


In [5]:
# make_factor_bases et synthesize_activations : importees de ict/factor_geometry.py
# (extraction #15943, corps deplaces a l'identique).
_bases_toy, _dim_toy = make_factor_bases(2, 8, mode="orthogonal")
_X_toy, _w_toy = synthesize_activations(_bases_toy, 16, rng=np.random.default_rng(0))
print(f"smoke: {len(_bases_toy)} bases {_bases_toy[0].shape} dim_per_factor={_dim_toy} X={_X_toy.shape}")
print("make_factor_bases et synthesize_activations pretes.")

smoke: 2 bases (8, 4) dim_per_factor=4 X=(16, 8)
make_factor_bases et synthesize_activations pretes.


## Exemple guidé 1 — Regime orthogonal : additivite exacte attendue

Avec des sous-espaces factoriels disjoints, l'additivite dimensionnelle devrait etre **exacte** : la dimension jointe egale la somme des dimensions factorielles. Ce test etablit la ligne de base.


In [6]:
results_orthogonal = {}
for seed in RNG_SEEDS:
    rng = np.random.default_rng(seed)
    bases, dim_per = make_factor_bases(N_FACTORS, N_DIM_RESIDUAL, mode="orthogonal", rng=rng)
    X, w = synthesize_activations(bases, N_TOKENS, signal_var=1.0, noise_var=0.0, rng=rng)
    mu, comps, sv, evr = weighted_pca(X, w)
    nc = nc_at(evr, NC_THRESHOLDS)
    # Bases factorielles individuelles pour comparaison
    factor_dims = []
    for B in bases:
        mu_i, comps_i, sv_i, evr_i = weighted_pca(X @ B @ B.T, w)  # projection sur facteur
        factor_dims.append(nc_at(evr_i, NC_THRESHOLDS)[95])
    sum_factors = sum(factor_dims)
    joint_dim = nc[95]
    # Overlap entre bases factorielles : devrait etre ~0 (orthogonalite parfaite)
    overlaps = []
    for i in range(N_FACTORS):
        for j in range(i + 1, N_FACTORS):
            M = basis_overlap(bases[i], bases[j])
            overlaps.append(float(M.max()))
    results_orthogonal[seed] = {
        "nc": nc,
        "joint_dim_95": joint_dim,
        "sum_factor_dims_95": sum_factors,
        "additivity_gap": abs(joint_dim - sum_factors),
        "max_pairwise_overlap": max(overlaps) if overlaps else 0.0,
    }

print("=== Exemple guide 1 : regime orthogonal ===")
for seed, r in results_orthogonal.items():
    print(f"  seed={seed} NC@95 joint={r['joint_dim_95']:>3} "
          f"sum_factors={r['sum_factor_dims_95']:>3} gap={r['additivity_gap']:>2} "
          f"max_pair_overlap={r['max_pairwise_overlap']:.4f}")


=== Exemple guide 1 : regime orthogonal ===
  seed=0 NC@95 joint=231 sum_factors=240 gap= 9 max_pair_overlap=0.0000
  seed=1 NC@95 joint=231 sum_factors=240 gap= 9 max_pair_overlap=0.0000
  seed=2 NC@95 joint=231 sum_factors=240 gap= 9 max_pair_overlap=0.0000
  seed=3 NC@95 joint=231 sum_factors=240 gap= 9 max_pair_overlap=0.0000
  seed=4 NC@95 joint=231 sum_factors=240 gap= 9 max_pair_overlap=0.0000


## Exemple guidé 2 — Regime superpose : orthogonalite degradee

Les sous-espaces se chevauchent partiellement (overlap 0.4). L'additivite devrait etre **degradee** : la dimension jointe est inferieure a la somme des dimensions factorielles (les facteurs partagent de la variance).


In [7]:
results_superposed = {}
for seed in RNG_SEEDS:
    rng = np.random.default_rng(seed)
    bases, dim_per = make_factor_bases(N_FACTORS, N_DIM_RESIDUAL, mode="superposed", overlap=0.4, rng=rng)
    X, w = synthesize_activations(bases, N_TOKENS, signal_var=1.0, noise_var=0.0, rng=rng)
    mu, comps, sv, evr = weighted_pca(X, w)
    nc = nc_at(evr, NC_THRESHOLDS)
    # Bases factorielles individuelles pour comparaison
    factor_dims = []
    for B in bases:
        mu_i, comps_i, sv_i, evr_i = weighted_pca(X @ B @ B.T, w)
        factor_dims.append(nc_at(evr_i, NC_THRESHOLDS)[95])
    sum_factors = sum(factor_dims)
    joint_dim = nc[95]
    overlaps = []
    for i in range(N_FACTORS):
        for j in range(i + 1, N_FACTORS):
            M = basis_overlap(bases[i], bases[j])
            overlaps.append(float(M.max()))
    results_superposed[seed] = {
        "nc": nc,
        "joint_dim_95": joint_dim,
        "sum_factor_dims_95": sum_factors,
        "additivity_gap": abs(joint_dim - sum_factors),
        "max_pairwise_overlap": max(overlaps) if overlaps else 0.0,
    }

print("=== Exemple guide 2 : regime superpose (overlap=0.4) ===")
for seed, r in results_superposed.items():
    print(f"  seed={seed} NC@95 joint={r['joint_dim_95']:>3} "
          f"sum_factors={r['sum_factor_dims_95']:>3} gap={r['additivity_gap']:>2} "
          f"max_pair_overlap={r['max_pairwise_overlap']:.4f}")


=== Exemple guide 2 : regime superpose (overlap=0.4) ===
  seed=0 NC@95 joint=155 sum_factors=240 gap=85 max_pair_overlap=0.6993
  seed=1 NC@95 joint=162 sum_factors=240 gap=78 max_pair_overlap=0.6941
  seed=2 NC@95 joint=174 sum_factors=240 gap=66 max_pair_overlap=0.8058
  seed=3 NC@95 joint=220 sum_factors=240 gap=20 max_pair_overlap=0.4946
  seed=4 NC@95 joint=216 sum_factors=240 gap=24 max_pair_overlap=0.3077


## Exemple guidé 3 — Sweep de bruit : sensibilite et verdicts falsifiables

Bruit additif gaussien de niveaux 0.0, 0.1, 0.25, 0.5, 1.0. La structure factorisee devrait disparaitre progressivement : au-dela d'un niveau de bruit pre-enregistre, les sous-espaces factoriels ne se separent plus de l'hypothese nulle (sous-espaces aleatoires apparies).


## Distinction belief-state vs factorisation géométrique

ICT-36 (ce notebook) et **ICT-37** (`ICT-37-FLens-BeliefState-Python.ipynb`, PR #15662 sur lane `myia-po-2023:CoursIA-2`) sont deux **modes complémentaires** de la même **Factored Geometry Lens (F-Lens)**. La distinction est **non triviale** et les résultats ne sont **pas interchangeables** :

| Question scientifique | Mode F-Lens | Verdict falsifiable |
|---|---|---|
| L'état prédictif du processus est-il **linéairement accessible** dans le residual stream ? | **belief-state** (ICT-37) | Probe R²/RMSE held-out ≥ seuil contre baseline shuffle/next-token. **Cible le théorème d'arXiv:2405.15943** (Shai et al.) |
| Les facteurs indépendants occupent-ils des **sous-espaces additifs**, superposés, orthogonaux ou entremêlés ? | **factored-geometry** (ICT-36) | NC@p + angles principaux + overlap + additivité dimensionnelle contre partitions aléatoires. **Cible le théorème d'arXiv:2602.02385** (Shai et al.) |

Un modèle peut avoir une **excellente décodabilité belief** (R² > 0.989, cf. ZM) **sans** factorisation orthogonale propre, et inversement. ICT-36 et ICT-37 rendent **deux verdicts distincts** sur le même réseau — c'est précisément ce que l'acceptance #15478 demande (« les résultats ne confondent pas décodabilité belief et factorisation géométrique »).

**Pour ce notebook** : on se concentre sur la géométrie des sous-espaces factoriels. La décodabilité belief n'est **pas** mesurée ici ; pour cela, voir ICT-37.


In [8]:
# null_overlap_distribution : importee du module (extraction #15943). Cf. l'avertissement
# de sa docstring -- le VRAI test H1 est ci-dessous (paires appariees null_pairs).
results_noise_sweep = {}
for noise_var in NOISE_LEVELS:
    per_seed = {}
    for seed in RNG_SEEDS:
        # Multiplicateur 1000 : avec 100, (seed=0, noise=1.0) et (seed=1, noise=0.0)
        # derivaient de la MEME graine (0*100+100 == 1*100+0) -- les conditions du
        # sweep n'etaient pas independantes. 1000 rend chaque paire (seed, noise) unique.
        rng = np.random.default_rng(seed * 1000 + int(noise_var * 100))
        bases, dim_per = make_factor_bases(N_FACTORS, N_DIM_RESIDUAL, mode="orthogonal", rng=rng)
        X, w = synthesize_activations(bases, N_TOKENS, signal_var=1.0, noise_var=noise_var, rng=rng)
        mu, comps, sv, evr = weighted_pca(X, w)
        # Mesure : NC@95 sur l'activation jointe vs somme des NC@95 factorielles
        nc_joint = nc_at(evr, NC_THRESHOLDS)[95]
        factor_nc = []
        for B in bases:
            mu_i, comps_i, sv_i, evr_i = weighted_pca(X @ B @ B.T, w)
            factor_nc.append(nc_at(evr_i, NC_THRESHOLDS)[95])
        sum_factors = sum(factor_nc)
        # Vrai test H1 : overlap entre les PROJECTIONS sur sous-espaces factoriels reels
        # vs overlap entre projections sur sous-espaces aleatoires apparies (meme dim).
        # Le test precedent comparait bases factorielles a une QR-aleatoire qui couvre
        # tout l'espace -- cela donne null95=1.0 systematiquement (artefact), pas un test.
        max_real = 0.0
        for i in range(N_FACTORS):
            for j in range(i + 1, N_FACTORS):
                # Projections sur les sous-espaces factoriels, puis PCA
                proj_i = X @ bases[i] @ bases[i].T
                proj_j = X @ bases[j] @ bases[j].T
                _, comps_i, _, _ = weighted_pca(proj_i, w)
                _, comps_j, _, _ = weighted_pca(proj_j, w)
                B_i = comps_i[:, :dim_per]
                B_j = comps_j[:, :dim_per]
                M = basis_overlap(B_i, B_j)
                if M.max() > max_real:
                    max_real = float(M.max())
        # Distribution nulle : paires de sous-espaces aleatoires apparies (meme dim_per_factor)
        null_pairs = []
        rng_null = np.random.default_rng(seed * 31 + 17)
        for _ in range(50):
            A = rng_null.standard_normal((N_DIM_RESIDUAL, dim_per))
            Q1, _ = np.linalg.qr(A)
            A = rng_null.standard_normal((N_DIM_RESIDUAL, dim_per))
            Q2, _ = np.linalg.qr(A)
            M = basis_overlap(Q1, Q2)
            null_pairs.append(float(M.max()))
        null_q95 = float(np.quantile(np.array(null_pairs), 0.95))
        per_seed[seed] = {
            "joint_dim_95": nc_joint,
            "sum_factor_dims_95": sum_factors,
            "additivity_gap": abs(nc_joint - sum_factors),
            "max_pair_overlap": max_real,
            "null_q95_overlap": null_q95,
            # Separation = overlap reel SOUS le quantile 95 de la distribution nulle.
            # Sens : sous-espaces factoriels reels ont un overlap plus PETIT que
            # des paires aleatoires apparies -- ils sont plus orthogonaux que le hasard.
            "separates_from_null": max_real < null_q95,
        }
    results_noise_sweep[noise_var] = per_seed

print("=== Exemple guide 3 : sweep de bruit ===")
print(f"{'noise':>6} {'seed':>4} {'joint':>6} {'sumF':>5} {'gap':>4} {'overlap':>7} {'null95':>7} {'sep?':>5}")
print(f"{'note':>6} sep? = overlap < null95 (sous-espaces plus orthogonaux que le hasard)")
for noise_var, per_seed in results_noise_sweep.items():
    for seed, r in per_seed.items():
        print(f"{noise_var:>6.2f} {seed:>4d} {r['joint_dim_95']:>6d} "
              f"{r['sum_factor_dims_95']:>5d} {r['additivity_gap']:>4d} "
              f"{r['max_pair_overlap']:>7.4f} {r['null_q95_overlap']:>7.4f} "
              f"{'YES' if r['separates_from_null'] else 'no':>5}")


=== Exemple guide 3 : sweep de bruit ===
 noise seed  joint  sumF  gap overlap  null95  sep?
  note sep? = overlap < null95 (sous-espaces plus orthogonaux que le hasard)
  0.00    0    231   240    9  0.0000  0.2609   YES
  0.00    1    231   240    9  0.0000  0.2662   YES
  0.00    2    231   240    9  0.0000  0.2628   YES
  0.00    3    231   240    9  0.0000  0.2761   YES
  0.00    4    231   240    9  0.0000  0.2654   YES
  0.10    0    231   240    9  0.0000  0.2609   YES
  0.10    1    231   240    9  0.0000  0.2662   YES
  0.10    2    231   240    9  0.0000  0.2628   YES
  0.10    3    231   240    9  0.0000  0.2761   YES
  0.10    4    231   240    9  0.0000  0.2654   YES
  0.25    0    231   240    9  0.0000  0.2609   YES
  0.25    1    231   240    9  0.0000  0.2662   YES
  0.25    2    231   240    9  0.0000  0.2628   YES
  0.25    3    231   240    9  0.0000  0.2761   YES
  0.25    4    231   240    9  0.0000  0.2654   YES
  0.50    0    231   240    9  0.0000  0.2609   YE

## Boucle fermee — modele entraine, plancher de Bayes et temoin non entraine

Les exemples guides ci-dessus mesurent la geometrie sur des activations **simulees** (`synthesize_activations`), ou la factorisation est posee par construction — donc retrouvee — et sur un temoin nul de sous-espaces **aleatoires**, qui ignore le biais inductif de l'architecture.

La lecture du protocole du depot de reference releve les deux manques : aucun des deux ne repond a l'objection « **un modele non entraine a peut-etre deja des sous-espaces separes** ».

Cette section ferme la boucle :

1. le **plancher de perte Bayes-optimal** du processus, calcule **exactement** sur la filtration forward du banc. Il sert de condition d'entree : aucune lecture geometrique n'est rendue si le modele n'a pas appris le processus ;
2. un modele **reellement entraine** sur le banc factorise (GRU a deux tetes, les memes processus Mess3 x RRXOR que l'ICT-37), et la geometrie lue sur **ses** activations ;
3. un **temoin nul par initialisations non entrainees de la meme architecture**, rapporte en percentiles 5/50/95.

Les trois lectures sont rapportees **a cote, jamais fusionnees** : qualite des sondes (decodabilite tenue hors echantillon), recouvrement spectral des sous-espaces de **coefficients** de ces sondes, et dimension additive. Le materiel vit dans `ict/factor_geometry_trained.py`, teste par `ict/tests/test_factor_geometry_trained.py`.


In [9]:
# Pre-enregistrement : cette cellule FIXE les predictions AVANT toute mesure.
# Protocole du commentaire du coordinateur du 2026-09-15.
PREDICTIONS = {
    "plancher": (
        "le modele entraine passe SOUS la perte uniforme ln(3)+ln(2) et reste proche du "
        "plancher de Bayes exact ; sinon aucune lecture geometrique n'est rendue"
    ),
    "dimensions": (
        "les deux facteurs rendent des sous-espaces de coefficients de dimensions "
        "distinctes : Mess3 vit dans un 2-simplexe (dim <= 2), RRXOR dans un "
        "4-simplexe (dim <= 4)"
    ),
    "additivite": (
        "additivity_gap == 0 si les facteurs sont additifs, strictement negatif s'ils "
        "partagent des directions. Prediction : ADDITIVE (les deux processus sont "
        "independants par construction du banc)"
    ),
    "temoin": (
        "le temoin non entraine doit DISCRIMINER : une metrique du modele entraine qui "
        "tombe dans la bande p05-p95 du temoin ne dit rien de ce qui a ete appris, et "
        "le verdict est INCONCLUSIF quelle que soit sa valeur"
    ),
}
for _k, _v in PREDICTIONS.items():
    print(f"  [pre-enregistre] {_k} : {_v}")


  [pre-enregistre] plancher : le modele entraine passe SOUS la perte uniforme ln(3)+ln(2) et reste proche du plancher de Bayes exact ; sinon aucune lecture geometrique n'est rendue
  [pre-enregistre] dimensions : les deux facteurs rendent des sous-espaces de coefficients de dimensions distinctes : Mess3 vit dans un 2-simplexe (dim <= 2), RRXOR dans un 4-simplexe (dim <= 4)
  [pre-enregistre] additivite : additivity_gap == 0 si les facteurs sont additifs, strictement negatif s'ils partagent des directions. Prediction : ADDITIVE (les deux processus sont independants par construction du banc)
  [pre-enregistre] temoin : le temoin non entraine doit DISCRIMINER : une metrique du modele entraine qui tombe dans la bande p05-p95 du temoin ne dit rien de ce qui a ete appris, et le verdict est INCONCLUSIF quelle que soit sa valeur


In [10]:
# Mesure : boucle fermee sur le banc factorise, puis temoin non entraine.
# Cout borne (~1 min CPU) : 400 pas sur 300 sequences de 100 pas.
from ict import bench_factorise as banc
from ict import factor_geometry_trained as fgt

bench = banc.FactoredBench(banc.Mess3Canonical(), banc.RRXOR())
LEN, N_TRAIN, N_VAL = 100, 300, 40
ech = bench.sample(LEN * (N_TRAIN + N_VAL) + LEN, seed_a=1, seed_b=1)
obs_joint = np.stack([ech["obs_a"], ech["obs_b"]], axis=1).astype(np.int64)

plancher_a = fgt.bayes_floor(bench.factor_a, ech["obs_a"][: LEN * N_TRAIN])
plancher_b = fgt.bayes_floor(bench.factor_b, ech["obs_b"][: LEN * N_TRAIN])
plancher = plancher_a + plancher_b
uniforme = float(np.log(3.0) + np.log(2.0))
print(f"plancher de Bayes exact : {plancher:.4f} nats "
      f"(Mess3 {plancher_a:.4f} + RRXOR {plancher_b:.4f})")
print(f"perte uniforme          : {uniforme:.4f} nats")

import torch as _torch

x = _torch.from_numpy(obs_joint)
x_train = x[: N_TRAIN * LEN].reshape(N_TRAIN, LEN, 2)
x_val = x[N_TRAIN * LEN : (N_TRAIN + N_VAL) * LEN].reshape(N_VAL, LEN, 2)

modele = fgt.FactoredRNN(seed=0)
res = fgt.train_model(modele, x_train, x_val, steps=400, eval_every=50)
print(f"\nmeilleure validation   : {res['best_val_nll']:.4f} nats au pas {res['best_step']}")
print(f"ecart au plancher       : {res['best_val_nll'] - plancher:+.4f} nats")
print(f"gate « le modele a appris le processus » : "
      f"{'PASS' if res['best_val_nll'] < uniforme else 'ECHEC'}")

actifs = fgt.collect_activations(modele, x_train)
bel_a = bench.factor_a.beliefs(ech["obs_a"][: LEN * N_TRAIN])
bel_b = bench.factor_b.beliefs(ech["obs_b"][: LEN * N_TRAIN])
rapport = fgt.factor_geometry_report(actifs, bel_a, bel_b)
temoin = fgt.untrained_null(lambda sd: fgt.FactoredRNN(seed=sd), x_train, bel_a, bel_b, n_init=6)

print("\nmetrique               entraine   temoin p05      p50      p95   lecture")
for _k in ("probe_r2_a", "probe_r2_b", "dim_a", "dim_b", "dim_union",
           "additivity_gap", "overlap_coef_max", "angle_coef_deg"):
    _st = temoin[_k]
    _dedans = _st["p05"] <= rapport[_k] <= _st["p95"]
    print(f"  {_k:20s} {rapport[_k]:+8.4f}   {_st['p05']:+8.4f} {_st['p50']:+8.4f} "
          f"{_st['p95']:+8.4f}   {'DANS la bande -> muet' if _dedans else 'hors bande'}")

MESURE = {"plancher": plancher, "uniforme": uniforme, "modele": rapport, "temoin": temoin,
          "best_val_nll": res["best_val_nll"], "best_step": res["best_step"]}


plancher de Bayes exact : 1.5448 nats (Mess3 1.0827 + RRXOR 0.4621)
perte uniforme          : 1.7918 nats



meilleure validation   : 1.5878 nats au pas 100
ecart au plancher       : +0.0429 nats
gate « le modele a appris le processus » : PASS



metrique               entraine   temoin p05      p50      p95   lecture
  probe_r2_a            +0.8469    +0.8250  +0.8273  +0.8442   hors bande
  probe_r2_b            +0.8753    +0.1442  +0.1604  +0.1788   hors bande
  dim_a                 +2.0000    +2.0000  +2.0000  +2.0000   DANS la bande -> muet
  dim_b                 +4.0000    +4.0000  +4.0000  +4.0000   DANS la bande -> muet
  dim_union             +6.0000    +6.0000  +6.0000  +6.0000   DANS la bande -> muet
  additivity_gap        +0.0000    +0.0000  +0.0000  +0.0000   DANS la bande -> muet
  overlap_coef_max      +0.2171    +0.2721  +0.3500  +0.3850   hors bande
  angle_coef_deg       +77.6254   +69.9738 +81.0826 +85.6761   DANS la bande -> muet


## Régimes génératifs : bruit du canal, structure conditionnelle (#20071)

La boucle fermée ci-dessus mesure un banc où les deux facteurs sont **indépendants et lisibles**. Les deux critères restants du suivi #15478 portent sur le **processus génératif lui-même**, pas sur les activations :

1. **Bruit sur le canal génératif.** L'objection « le bruit additif sur les activations ne dit rien du processus » se retourne : ici le bruit porte sur les **émissions du banc** (matrice `E` pour le facteur Moore, tenseur d'arêtes `W` pour le Mealy), **transitions intactes**. Chaque ligne d'émission est méprisée vers l'uniforme d'un facteur `gamma` : à `gamma = 1` les émissions sont muettes sur l'état caché et le plancher de Bayes vaut **exactement** l'entropie uniforme. Le critère exige **un entraînement complet par niveau de bruit**, contre le **même témoin d'initialisations non entraînés rejoué à chaque niveau** — un témoin mesuré à un seul niveau ne borne rien.
2. **Structure générative conditionnelle.** Le **token du facteur parent** sélectionne la **variante d'émission du facteur enfant** (`E_v = E + kappa * roll(Delta, v)`, rotations cycliques d'une perturbation à lignes de somme nulle), à **marginales appariées exactement** : les `q` rotations s'annulent, la marginal de l'enfant — même prise conditionnellement au token parent — est identique à celle du banc indépendant. Toute la dépendance vit dans la loi **jointe**. La question falsifiable : la géométrie lue sur le modèle entraîné diffère-t-elle du banc indépendant **à marginales égales** ?

Les constructions vivent dans `ict/bench_factorise.py` (`NoisedChannel`, `ConditionalBench`, `ConditionalStateBench`, `conditional_child_floor`), testées par `ict/tests/test_bench_regimes_generatifs.py` — dont la contre-vérification des filtres par **énumération brute des chemins d'états** (filtre par préfixe : le chemin complet donnerait le lisse, une quantité différente).

### Les deux premières passes, et un artefact committé

La cellule suivante pré-enregistre la **première passe** (5 niveaux, 400 pas, une graine modèle). Ses deux lectures ont montré que le protocole ne pouvait pas trancher les falsifiables, et chacune pour une raison mécanique :

- **régime 1** : la géométrie n'était rendue qu'aux niveaux où le gate passe — or le critère se joue précisément là où le modèle n'apprend plus rien, c'est-à-dire là où un modèle à la perte plate doit se confondre avec les initialisations non entraînées ;
- **régime 2** : l'`additivity_gap` est un **entier au rang près** ; sur `Mess3 x Mess3` le témoin rend la même valeur aux six initialisations, donc une bande de largeur **nulle**, qui ne peut discriminer aucun modèle.

La **seconde passe** corrige les deux : 7 niveaux, 1200 pas, témoin rejoué à **chaque** niveau, lecture rendue à **tous** les niveaux, 3 graines modèle, et une métrique **continue** (`overlap_energy`, ajoutée au rapport par ce suivi). Son protocole est figé dans le docstring de `ict/factor_geometry_regimes.py` **avant** son exécution ; la campagne y est livrée comme **artefact committé** (`ict/results/ict36_regime*.json`, pattern du banc ICT-25) et les deux cellules de code ci-dessous **lisent l'artefact** — elles ne rejouent pas ~45 min de CPU. Régénération : `python -m ict.factor_geometry_regimes --steps 1200`.

### Troisième passe : deux critères rendus décidables

Les deux passes ci-dessus ont produit des lectures **honnêtes mais muettes**, chacune pour une raison mécanique distincte. Les trois obstacles que la troisième passe lève sont nommés ci-dessous, avec leur mesure.

- **La métrique nommée par l'issue n'est pas invariante.** `overlap_coef_max` est le maximum des **éléments** de `Ba^T Bb` : deux bases du même plan donnent 1,000 puis 0,707 après une rotation, sans que la géométrie du sous-espace bouge. Une courbe qui bouge sous l'effet d'un choix de base ne peut pas trancher un critère. La troisième passe ajoute `overlap_principal_max` — le plus grand **singulier** de `Ba^T Bb`, c'est-à-dire le cosinus principal maximal — qui est invariant, et rend l'issue des **trois** métriques (`overlap_coef_max`, `overlap_principal_max`, `overlap_energy`) dans l'artefact : `retour_durable` (avec le seuil nommé), `jamais_sorti` ou `indécidable`. Les deux lectures peuvent désaccorder ; le désaccord est affiché plutôt que masqué par un choix silencieux.
- **Le contrôle « indépendant » ne l'était pas.** Deux générateurs de **même classe effective** tirés avec la **même graine** partagent leur trajectoire d'états cachés : mesure sur `Mess3 x Mess3` (60 000 pas), l'information mutuelle entre les deux observations vaut **0,677 nats** à graine identique, contre **0,0001** à graines distinctes, et 0,000002 pour `Mess3 x RRXOR` à graine identique (cette dernière classe consomme le flux d'uniformes autrement). Les campagnes précédentes samplaient les deux facteurs avec `seed_a = seed_b = data_seed` : le contrôle était donc le banc **le plus entrelacé** des deux, et l'écart observé était l'**inverse** de la sémantique voulue. Désormais les graines sont distinctes, le banc **refuse** ce cas pathologique, et l'indépendance du contrôle est **mesurée et publiée** dans l'artefact (`independance_facteurs`).
- **Une cible dégénérée et une bande de largeur nulle se lisaient comme des mesures.** À `gamma = 1` (canal muet) la cible est constante au bit près (amplitude 2,8e-17 à 5,6e-17) : les résidus d'arrondi suffisaient à faire sortir une direction d'une SVD, d'où un R^2 **fabriqué** de 0,747 et un rang lu de 1. Un plancher de variance de cible rend désormais le rang nul et la métrique `nan` (le JSON porte `null`). De même, l'`additivity_gap` est un **entier au rang près** : sa bande de témoin est `[0 ; 0]`, de largeur nulle, et une valeur qui y entre n'y entre que par égalité. La position d'une métrique contre une telle bande est rendue `n/a` — une bande qui ne discrimine rien ne se dit pas « dans la bande » — et le critère 2 compare **malgré** la bande dégénérée, puisque c'est le **signe** du gap qui est demandé.

### Troisième régime : une structure conditionnelle NÉCESSAIRE

Les deux premiers régimes mesuraient un banc où la dépendance **n'est pas nécessaire** : dans le banc à variante d'émission, la part de variance de la cible expliquée par le token du parent vaut **-0,0002**, donc un modèle qui **ignore** la dépendance est aussi bon sur la cible. Un `additivity_gap` nul y est la **bonne réponse de ce banc** : c'est le banc qui manquait, pas la métrique, et aucun budget d'entraînement n'y changerait quoi que ce soit.

Le troisième régime (`ConditionalStateBench`) pose l'état caché de l'enfant comme une **rotation du token observé** du parent : `s_enfant[t] = (token[t] + decalage) mod q`. La rotation est une bijection, donc les marginales de l'enfant restent appariées au contrôle (écart théorique **0,0**, empirique 0,0034 à 20 000 pas), et la lecture de l'enfant n'a plus d'autre route que la direction du parent — R^2(token -> cible) = **+1,0000**, information mutuelle(token ; état) = **log 3 = 1,0986 nats**, le plafond. Le critère 2 devient enfin testable : sur un modèle qui encode la dépendance, les sous-espaces de lecture des deux facteurs se recouvrent, et l'écart d'additivité devient **négatif**.

Les quatre artefacts de la troisième passe (`ict36_regime1_sweep_bruit_canal.json`, `ict36_regime2_conditionnel.json`, `ict36_regime2b_necessite.json`, `ict36_regime2b_conditionnel_etat.json`) sont lus par les cellules qui suivent ; ils portent les issues des deux critères, **calculées par le code** et non lues à l'oeil.


In [11]:
# Pre-enregistrement de la PREMIERE passe (5 niveaux, 400 pas, une graine modele),
# conserve comme trace : ces predictions sont FIXEES avant toute mesure des deux
# regimes et n'ont PAS etre reecrites apres coup. Le protocole de la SECONDE passe
# (7 niveaux, 1200 pas, 3 graines, lecture rendue a tous les niveaux, temoin par
# niveau) est fige, lui, dans le docstring de ict/factor_geometry_regimes.py AVANT
# l'execution de la campagne -- c'est l'objet de l'artefact lu plus bas.
PREDICTIONS_REGIMES = {
    "bruit_canal": (
        "le plancher de Bayes joint se releve de facon monotone avec gamma, jusqu'a "
        "ln(3)+ln(2) EXACT a gamma=1 (emissions muettes) ; a chaque niveau ou le gate "
        "passe, la geometrie reste lisible avec des dimensions stables (les etats caches "
        "n'ont pas bouge) ; a gamma=1 le gate ECHOUE et aucune lecture n'est rendue -- "
        "le critere exige l'entrainement complet par niveau, pas la reussite a chaque niveau"
    ),
    "structure_conditionnelle": (
        "les marginales par canal sont identiques au banc independant (appariement "
        "exact), donc le canal A n'est pas affecte ; pour le canal B, le modele ne voit "
        "PAS le token parent de l'instant qu'il predit (il ne connait que le passe "
        "commun) : sa perte tombe ENTRE le plancher aveugle aux variants et le plancher "
        "genie (variante observee). Toute difference geometrique avec le banc "
        "independant est une difference de loi jointe, pas de marginales"
    ),
    "temoin_par_niveau": (
        "le temoin non entraine est rejoue a CHAQUE niveau de bruit : sa bande bouge "
        "avec gamma, et une metrique du modele entraine ne se lit qu contre la bande "
        "du MEME niveau"
    ),
}
for _k, _v in PREDICTIONS_REGIMES.items():
    print(f"  [pre-enregistre] {_k} : {_v}")

  [pre-enregistre] bruit_canal : le plancher de Bayes joint se releve de facon monotone avec gamma, jusqu'a ln(3)+ln(2) EXACT a gamma=1 (emissions muettes) ; a chaque niveau ou le gate passe, la geometrie reste lisible avec des dimensions stables (les etats caches n'ont pas bouge) ; a gamma=1 le gate ECHOUE et aucune lecture n'est rendue -- le critere exige l'entrainement complet par niveau, pas la reussite a chaque niveau
  [pre-enregistre] structure_conditionnelle : les marginales par canal sont identiques au banc independant (appariement exact), donc le canal A n'est pas affecte ; pour le canal B, le modele ne voit PAS le token parent de l'instant qu'il predit (il ne connait que le passe commun) : sa perte tombe ENTRE le plancher aveugle aux variants et le plancher genie (variante observee). Toute difference geometrique avec le banc independant est une difference de loi jointe, pas de marginales
  [pre-enregistre] temoin_par_niveau : le temoin non entraine est rejoue a CHAQUE niveau

In [12]:
# Regime 1 -- TROISIEME passe, lecture de l'artefact (protocole v2, #20071) :
# 7 niveaux, 1200 pas, temoin rejoue a CHAQUE niveau, et lecture geometrique
# rendue a TOUS les niveaux -- gate reussi ou echoue. C'est la difference
# decisive avec la premiere passe : le critere falsifiable se joue la ou le
# modele n'apprend plus rien. La campagne vit dans ict/factor_geometry_regimes.py
# (testee), livree en artefact commite -- cette cellule LIT, elle ne rejoue pas
# ~45 min de CPU (pattern artefacts-commits du banc ICT-25).
# Regeneration : python -m ict.factor_geometry_regimes --steps 1200
# Nit Hermes #21652 (2026-10-21) : la colonne `dims` est IMPRIMEE, pas seulement
# collectee dans LIGNES_V2 -- la lecture du regime 1 cite `(2, 4, 6)` et
# `(1, 1, 2)`, et le lecteur du carnet doit pouvoir l'ancre au tableau, sans
# ouvrir l'artefact JSON.
# Troisieme passe : un niveau DEGENERE (canal muet a gamma = 1, cible constante
# au bit pres) rend un rang nul et des metriques non definies -- l'artefact
# JSON les porte donc comme `null` (le JSON strict refuse NaN). Elles se
# RAPPORTENT (`n/a`), elles ne se lisent pas comme des nombres, et surtout elles
# ne font pas tomber la cellule.
import json

from ict.factor_geometry_regimes import format_ou_na, position_contre_bande

ART1 = ICT_ROOT / "ict" / "results" / "ict36_regime1_sweep_bruit_canal.json"
art1 = json.loads(ART1.read_text(encoding="utf-8"))
p1 = art1["protocole"]


def _bande(b):
    """La bande p05-p95 d'une metrique, ou `degeneree` si elle ne dit rien."""
    if not b.get("n"):
        return "degeneree"
    return f"{format_ou_na(b['p05'])}-{format_ou_na(b['p95'])}"


print(f"protocole v2 : {p1['steps']} pas, L={p1['length']}, {p1['n_train']} sequences "
      f"d'entrainement / {p1['n_val']} de validation, temoin sur {p1['n_init']} initialisations")
print()
print(f"{'gamma':>5}  {'plancher':>9}  {'val':>8}  {'gate':>5}  {'r2_a':>5} {'r2_b':>5}  "
      f"{'ovl':>5} {'bande ovl':>13}  {'position':>11}  {'energie':>7} {'bande E':>13}  "
      f"{'dims':>12}")
LIGNES_V2 = []
for l in art1["niveaux"]:
    r, t = l["rapport"], l["temoin"]
    pos = position_contre_bande(r["overlap_coef_max"], t["overlap_coef_max"])
    pos_e = position_contre_bande(r["overlap_energy"], t["overlap_energy"])
    dims = f"({r['dim_a']:.0f},{r['dim_b']:.0f},{r['dim_union']:.0f})"
    print(f"{l['gamma']:5.2f}  {l['plancher_joint']:9.5f}  {l['best_val_nll']:8.4f}  "
          f"{'PASS' if l['gate'] else 'ECHEC':>5}  {format_ou_na(r['probe_r2_a'], '{:5.3f}')} "
          f"{format_ou_na(r['probe_r2_b'], '{:5.3f}')}  "
          f"{format_ou_na(r['overlap_coef_max'], '{:5.3f}')} [{_bande(t['overlap_coef_max']):>13}]  "
          f"{pos:>11}  {format_ou_na(r['overlap_energy'], '{:7.3f}')} "
          f"[{_bande(t['overlap_energy']):>13}]  {dims:>12}")
    LIGNES_V2.append({"gamma": l["gamma"], "gate": l["gate"], "pos_ovl": pos,
                      "pos_energie": pos_e,
                      "dims": (r["dim_a"], r["dim_b"], r["dim_union"]),
                      "plancher": l["plancher_joint"], "val": l["best_val_nll"]})

# --- verdict du falsifiable, CALCULE sur l'artefact (pas lu a l'oeil) ---
planchers_v2 = [x["plancher"] for x in LIGNES_V2]
monotone = all(b >= a - 1e-9 for a, b in zip(planchers_v2, planchers_v2[1:]))
positions = [x["pos_ovl"] for x in LIGNES_V2]
hors = sum(1 for x in positions if x not in ("dans_bande", "n/a"))
print()
print(f"plancher monotone en gamma : {monotone} -- gamma=1 mesure {planchers_v2[-1]:.5f} "
      f"contre ln(3)+ln(2) = {float(np.log(3.0) + np.log(2.0)):.5f}")
print(f"positions de overlap_coef_max contre la bande de SON niveau : {positions}")
print(f"-> {hors} niveaux sur {len(positions)} HORS bande ; lecture rendue aux "
      f"{len(LIGNES_V2)} niveaux dont {sum(1 for x in LIGNES_V2 if not x['gate'])} a gate ECHEC")

# --- les TROIS verdicts du critere 1, calcules par le module ------------------
# L'issue nomme `overlap_coef_max`, qui n'est PAS invariante par rotation des
# bases : une courbe qui bouge sous l'effet d'un choix de base ne peut pas
# trancher un critere. Les trois metriques sont donc verifiees, et leur
# eventuel desaccord est affiche au lieu d'etre masque par un choix silencieux.
print()
print("verdicts du critere 1, calcules dans l'artefact :")
for nom, v in art1.get("verdicts", {}).items():
    seuil = v.get("seuil")
    print(f"  {nom:22s} : {v['issue']}"
          + (f"  (seuil gamma = {seuil})" if seuil is not None else ""))
MESURE_BRUIT_CANAL_V2 = LIGNES_V2


protocole v2 : 1200 pas, L=100, 300 sequences d'entrainement / 40 de validation, temoin sur 6 initialisations

gamma   plancher       val   gate   r2_a  r2_b    ovl     bande ovl     position  energie       bande E          dims
 0.00    1.54484    1.5850   PASS  0.803 0.896  0.164 [  0.274-0.373]   sous_bande    0.030 [  0.100-0.211]       (2,4,6)
 0.25    1.75613    1.7752   PASS  0.872 0.090  0.190 [  0.313-0.360]   sous_bande    0.088 [  0.106-0.180]       (2,4,6)
 0.50    1.78900    1.7927  ECHEC  0.823 0.033  0.397 [  0.285-0.351]  dessus_bande    0.138 [  0.103-0.171]       (2,4,6)
 0.65    1.79138    1.7926  ECHEC  0.814 0.022  0.280 [  0.235-0.354]   dans_bande    0.087 [  0.072-0.188]       (2,4,6)
 0.75    1.79173    1.7919  ECHEC  0.809 0.031  0.269 [  0.275-0.359]   sous_bande    0.115 [  0.093-0.177]       (2,4,6)
 0.90    1.79177    1.7930  ECHEC  0.807 0.520  0.340 [  0.293-0.415]   dans_bande    0.151 [  0.107-0.203]       (2,4,6)
 1.00    1.79176    1.7924  ECHEC  n/a

### Lecture du régime 1 (bruit du canal) — troisième passe

Le protocole est celui de la seconde passe (1200 pas, témoin rejoué à chaque niveau, lecture rendue aux sept niveaux, gate réussi ou échoué) ; la troisième passe change ce qui est **rendu**, pas ce qui est mesuré. Le niveau muet `gamma = 1` est désormais rapporté comme **dégénéré** : cible constante au bit près, plancher de variance de cible, rang nul (dims `(0, 0, 0)`), métriques `n/a` — la fausse remontée de `r2_b` à 0.743 sur cible constante, que la seconde passe rapportait déjà avec suspicion, est fermée par construction. Et l'issue des **trois** métriques est **calculée par le module** et lue telle quelle.

- le **plancher de Bayes se relève** avec `gamma`, de 1.54484 à 1.79176 = `ln(3) + ln(2)` exactement : la dynamique latente est intacte, le canal est muet — le pré-enregistrement est vérifié sur ce point (l'unique écart à la monotonie, 1.79177 -> 1.79176 au dernier pas, est un effet de taille d'échantillon déjà rapporté par la seconde passe) ;
- `r2_b` s'effondre dès `gamma = 0.25` (0.896 -> 0.090) : le canal enfant devient illisible au sens de la prédiction de sa cible, alors que `r2_a` tient (0.803 à 0.872 sur tous les niveaux lisibles). La remontée à 0.520 à `gamma = 0.9` est rapportée sans être interprétée ;
- les **verdicts du critère 1, calculés dans l'artefact** : `overlap_coef_max` — la métrique nommée par l'issue — rend **`retour_durable`, seuil `gamma* = 0.65`** : la courbe sort de la bande à un seul niveau (`gamma = 0.50` : 0.397 contre `[0.285, 0.351]`) et n'en ressort plus au-delà de 0.65 (`gamma = 0.65` : 0.280 dans `[0.235, 0.354]` ; `gamma = 0.90` : 0.340 dans `[0.293, 0.415]`). Le niveau `gamma = 0.75` (0.269 contre `[0.275, 0.359]`) est **marginalement sous** sa bande, mais le seuil ne repose pas sur lui : 0.65 et 0.90 sont **dans** leur bande. Le niveau `gamma = 1` est **dégénéré** : il ne compte ni comme sortie ni comme retour, et il n'empêche donc pas le verdict. Les deux métriques qui ne bougent pas sous une rotation des bases rendent **`jamais_sorti`** : `overlap_principal_max` et `overlap_energy` restent sous ou dans la bande de leur témoin à tous les niveaux lisibles.

**Ce que le verdict permet de conclure, et ce qu'il ne permet pas.** Les **deux branches** du falsifiable sont désormais observées, mais sur **deux lectures différentes de la même géométrie**. La branche « n'en sort jamais » est établie sur les deux métriques invariantes : à aucun niveau de bruit le recouvrement des sous-espaces de lecture du modèle entraîné ne s'est séparé par le haut de la bande des initialisations non entraînées — même là où le modèle apprend encore (`r2_a` > 0.8 jusqu'à `gamma = 0.9`). La branche « retombe dans la bande au-delà d'un seuil » est établie sur la métrique nommée par l'issue, avec le seuil nommé (`gamma* = 0.65`). Le désaccord entre métriques est un résultat : il est affiché, pas masqué — et il se lit. La courbe n'est pas monotone : le modèle entraîné est **sous** la bande de son témoin aux deux premiers niveaux (0.164 contre `[0.274, 0.373]`, puis 0.190 contre `[0.313, 0.360]`), ce qui dit qu'il est plus factorisé que des initialisations non entraînées ; l'écart de `gamma = 0.50` le ramène au-dessus, et il **disparaît dès que la métrique est invariante** — une grandeur qui s'évanouit sous rotation des bases ne peut pas être la signature d'un processus génératif. Ce que cela **ne** permet pas de dire : (i) que le modèle n'apprend rien — il lit le facteur Moore jusqu'au bout ; (ii) que l'écart de `gamma = 0.50` est un effet du bruit sur la géométrie, puisque les deux métriques invariantes ne le voient pas.


In [13]:
# Regime 2 -- TROISIEME passe, lecture de l'artefact : banc CONDITIONNEL (variante
# d'emission) contre controle INDEPENDANT a marginales appariees, 3 graines modele,
# 1200 pas, kappa pousse vers sa borne de stochasticite. Deux metriques de
# recouvrement sont lues contre leur temoin du MEME banc :
#   - additivity_gap, ENTIER au rang pres (sature sur ces bancs) ;
#   - overlap_energy, CONTINUE (bande de temoin de largeur non nulle).
# Troisieme passe : le controle est desormais **mesure** independant (les
# facteurs sont tires avec des graines DISTINCTES -- meme classe effective +
# meme graine = trajectoire d'etats cachees partagee, 0,677 nats mesures), la
# metrique d'overlap **invariante** `overlap_principal_max` s'y ajoute, et le
# VERDICT du critere 2 est calcule par le module et lu tel quel.
import json

from ict.factor_geometry_regimes import format_ou_na, position_contre_bande

art2 = json.loads((ICT_ROOT / "ict" / "results" / "ict36_regime2_conditionnel.json")
                  .read_text(encoding="utf-8"))
pl = art2["planchers"]
ap = art2["appariement"]
ind = art2.get("independance_facteurs", {})
print(f"kappa={art2['kappa']}  {art2['steps']} pas  graines modele={art2['model_seeds']}  "
      f"temoin={art2['n_init']} initialisations")
print(f"appariement des marginales : matrices exactes = {ap['matrices_exact']} "
      f"(ecart empirique max {ap['ecart_empirique_max']:.4f})")
print(f"independance MESUREE des facteurs : conditionnel {ind.get('conditionnel', float('nan')):.6f} nats / "
      f"independant {ind.get('independant', float('nan')):.6f} nats")
print(f"planchers enfant : genie {pl['genie']:.4f} / aveugle {pl['aveugle']:.4f} "
      f"-> prime conditionnelle {pl['prime_conditionnelle']:+.4f} nats")
print(f"plancher parent {pl['parent']:.4f}  |  uniforme joint {pl['uniforme_joint']:.4f} (= 2 ln 3)")
print()
for nom in ("conditionnel", "independant"):
    bloc = art2["resultats"][nom]
    tg, te = bloc["temoin"]["additivity_gap"], bloc["temoin"]["overlap_energy"]
    print(f"[{nom}]  bande temoin : gap [{format_ou_na(tg['p05'], '{:+.3f}')}, {format_ou_na(tg['p95'], '{:+.3f}')}] "
          f"(largeur {format_ou_na((tg['p95'] - tg['p05']) if tg.get('n') else None, '{:.3f}')})  "
          f"energie [{format_ou_na(te['p05'])}, {format_ou_na(te['p95'])}]")
    for seed, res in bloc["par_seed"].items():
        r = res["rapport"]
        pg = position_contre_bande(r["additivity_gap"], tg)
        pe = position_contre_bande(r["overlap_energy"], te)
        print(f"  seed {seed}: val={res['best_val_nll']:.4f} gate={'PASS' if res['gate'] else 'ECHEC'}  "
              f"gap={format_ou_na(r['additivity_gap'], '{:+.3f}')} [{pg}]  "
              f"energie={format_ou_na(r['overlap_energy'], '{:.3f}')} [{pe}]  "
              f"ovl_coef={format_ou_na(r['overlap_coef_max'])}  "
              f"ovl_principal={format_ou_na(r.get('overlap_principal_max'))}  "
              f"dims=({r['dim_a']:.0f},{r['dim_b']:.0f},{r['dim_union']:.0f})")
    print()
ENERGIES_COND = [bloc_par["rapport"]["overlap_energy"]
                 for bloc_par in art2["resultats"]["conditionnel"]["par_seed"].values()]
ENERGIES_INDEP = [bloc_par["rapport"]["overlap_energy"]
                  for bloc_par in art2["resultats"]["independant"]["par_seed"].values()]
print(f"separation des energies : conditionnel [{min(ENERGIES_COND):.3f}, {max(ENERGIES_COND):.3f}] "
      f"vs independant [{min(ENERGIES_INDEP):.3f}, {max(ENERGIES_INDEP):.3f}] "
      f"-- recouvrement des intervalles : {max(ENERGIES_COND) >= min(ENERGIES_INDEP)}")

# --- VERDICT du critere 2, CALCULE par le module (pas lu a l'oeil) -----------
v2 = art2.get("verdict_critere_2")
if v2:
    print()
    print(f"verdict du critere 2 (calcule) -- marginales appariees : {v2['marginales_appariees']}")
    for nom, b in v2["bras"].items():
        print(f"  {nom:14s} : {'SATISFAIT' if b['satisfait'] else 'non satisfait'}  "
              f"(gap negatif : {b['gap_negatif']}, hors bande : {b['hors_bande']}, "
              f"gaps : {[g['additivity_gap'] for g in b['gaps']]})")
MESURE_CONDITIONNEL_V2 = {"energies_cond": ENERGIES_COND, "energies_indep": ENERGIES_INDEP}


kappa=0.28  1200 pas  graines modele=[0, 1, 7]  temoin=6 initialisations
appariement des marginales : matrices exactes = True (ecart empirique max 0.0029)
independance MESUREE des facteurs : conditionnel 0.000172 nats / independant 0.000153 nats
planchers enfant : genie 0.9281 / aveugle 1.0971 -> prime conditionnelle +0.1690 nats
plancher parent 1.0827  |  uniforme joint 2.1972 (= 2 ln 3)

[conditionnel]  bande temoin : gap [+0.000, +0.000] (largeur 0.000)  energie [0.018, 0.056]
  seed 0: val=2.1878 gate=PASS  gap=+0.000 [n/a]  energie=0.027 [dans_bande]  ovl_coef=0.181  ovl_principal=0.211  dims=(2,2,4)
  seed 1: val=2.1925 gate=PASS  gap=+0.000 [n/a]  energie=0.078 [dessus_bande]  ovl_coef=0.336  ovl_principal=0.391  dims=(2,2,4)
  seed 7: val=2.1873 gate=PASS  gap=+0.000 [n/a]  energie=0.030 [dans_bande]  ovl_coef=0.203  ovl_principal=0.232  dims=(2,2,4)

[independant]  bande temoin : gap [+0.000, +0.000] (largeur 0.000)  energie [0.007, 0.098]
  seed 0: val=2.1897 gate=PASS  gap=+

### Lecture du régime 2 (structure conditionnelle) -- troisième passe

La troisième passe ne change pas le banc ; elle change le **contrôle**. Les passes précédentes samplaient les deux facteurs avec la même graine : deux générateurs de même classe effective à graine identique partagent leur trajectoire d'états cachés, et le « contrôle indépendant » était en réalité le banc le plus entrelacé des deux (0.677 nats d'information mutuelle mesurés, contre ~0.0002 à graines distinctes). Désormais les graines sont distinctes et l'indépendance du contrôle est **mesurée dans l'artefact** : `independance_facteurs` rend 0.000172 nats (banc conditionnel) et 0.000153 nats (contrôle indépendant).

- l'**appariement reste exact** (matrices exactes, écart empirique maximal 0.0029 sur l'échantillon) et la **prime d'information conditionnelle est mesurée** : +0.1690 nats entre le filtre génie (0.9281) et le filtre aveugle (1.0971). À marginales identiques, connaître le token parent vaut ~0.17 nats par symbole du canal enfant ;
- en **perte, l'avantage de la seconde passe a disparu** : le conditionnel rend 2.1878 / 2.1925 / 2.1873 et le contrôle indépendant 2.1897 / 2.1900 / 2.1881 -- une graine sur trois rend le conditionnel au-dessus du contrôle. La séparation « stable aux trois graines » (~0.017 nats) de la seconde passe était un artefact du contrôle entrelacé : elle est **retirée**, et c'est exactement ce que la mesure d'indépendance imposait ;
- l'**`overlap_energy` ne sépare plus les bancs** : conditionnel `[0.027, 0.078]` contre indépendant `[0.012, 0.090]`, les intervalles se recouvrent. La séparation sans recouvrement de la seconde passe était le même artefact ;
- l'**`additivity_gap` reste saturé** à +0.000 aux trois graines, dans une bande de témoin de largeur nulle. Le verdict du critère 2 est calculé par le module et rendu tel quel : **non satisfait** dans les deux bras (gap jamais négatif, jamais hors bande).

**Ce que cela tranche, et ce que cela ne tranche pas.** Le contrôle étant désormais mesuré indépendant, la lecture honnête est : **aucun instrument de ce banc ne sépare le modèle entraîné sur la loi jointe du contrôle à marginales égales** -- ni en perte, ni par le recouvrement des sous-espaces de lecture. La dépendance existe pourtant (prime génie +0.169 nats), mais elle est invisible pour ces instruments sur un alphabet commun ; le critère 2 reste ouvert sur ce banc, et la question est dévolue au régime suivant, où la dépendance est cette fois **nécessaire** à la cible. Ce que le résultat ne permet pas de dire : que le modèle « a appris » ou « n'a pas appris » la loi jointe -- sur trois graines, la perte du conditionnel se confond avec celle du contrôle, l'échantillon ne distingue pas.

In [14]:
# Regime 2b -- TROISIEME passe : la structure conditionnelle rendue NECESSAIRE.
# La campagne lit deux artefacts :
#   - ict36_regime2b_necessite.json : la mesure CPU de necessite (aucun
#     entrainement) : le token du parent explique-t-il la CIBLE ?
#   - ict36_regime2b_conditionnel_etat.json : l'entrainement du modele sur le
#     banc ou l'etat cache de l'enfant est une ROTATION du token observe,
#     contre un controle independant aux MEMES marginales.
import json

from ict.factor_geometry_regimes import format_ou_na, position_contre_bande

art_nec = json.loads((ICT_ROOT / "ict" / "results" / "ict36_regime2b_necessite.json")
                     .read_text(encoding="utf-8"))
art2b = json.loads((ICT_ROOT / "ict" / "results" / "ict36_regime2b_conditionnel_etat.json")
                    .read_text(encoding="utf-8"))

print("=== 1. NECESSITE : le token du parent explique-t-il la cible ? ===")
print(f"n = {art_nec['n']} pas, graine de donnees {art_nec['data_seed']}, "
      f"decalage {art_nec['shift']}")
print(f"{'regime':18s} {'R2(token->cible)':>16s} {'var(cible)':>12s} "
      f"{'I(tok;cible)':>13s} {'I(tok;obs)':>12s}")
for nom, m in art_nec["mesures"].items():
    print(f"{nom:18s} {m['r2_token_belief_b']:+16.4f} {m['variance_belief_b']:12.6f} "
          f"{m['prime_belief']:+13.4f} {m['prime_observation']:+12.4f}")
ap = art_nec["appariement"]
print(f"appariement des marginales : exact = {ap['marginales_exactes']}  "
      f"ecart theorique max {ap['ecart_theorique_max']:.2e}  "
      f"ecart empirique max {ap['ecart_empirique_max']:.4f}")

print()
print("=== 2. ENTRAINEMENT : le gap sort-il de la bande du temoin ? ===")
print(f"decalage = {art2b['shift']}  {art2b['steps']} pas  graines modele = {art2b['model_seeds']}  "
      f"temoin = {art2b['n_init']} initialisations")
ind = art2b["independance_facteurs"]
print(f"independance MESUREE des facteurs : conditionnel {ind['conditionnel_etat']:.6f} nats / "
      f"independant {ind['independant']:.6f} nats")
for nom, ent in art2b["resultats"].items():
    tg, te = ent["temoin"]["additivity_gap"], ent["temoin"]["overlap_energy"]
    print(f"[{nom}]  bande gap [{format_ou_na(tg['p05'], '{:+.3f}')}, {format_ou_na(tg['p95'], '{:+.3f}')}]  "
          f"energie [{format_ou_na(te['p05'])}, {format_ou_na(te['p95'])}]  "
          f"R2(token->cible) = {ent['necessite']['r2_token_belief_b']:+.4f}")
    for seed, res in ent["par_seed"].items():
        r = res["rapport"]
        # Le critere 2 demande le SIGNE du gap : la bande du temoin est de
        # largeur nulle (entier au rang pres), elle ne peut donc etre franchie
        # que par le signe -- d'ou `comparer_bande_degeneree=True`.
        pg = position_contre_bande(r["additivity_gap"], tg, comparer_bande_degeneree=True)
        pe = position_contre_bande(r["overlap_energy"], te)
        print(f"  seed {seed}: val={res['best_val_nll']:.4f} gate={'PASS' if res['gate'] else 'ECHEC'}  "
              f"gap={format_ou_na(r['additivity_gap'], '{:+.3f}')} [{pg}]  "
              f"energie={format_ou_na(r['overlap_energy'], '{:.3f}')} [{pe}]  "
              f"ovl_principal={format_ou_na(r.get('overlap_principal_max'))}  "
              f"dims=({r['dim_a']:.0f},{r['dim_b']:.0f},{r['dim_union']:.0f})")

print()
print("=== 3. VERDICT du critere 2 (calcule dans l'artefact, pas lu a l'oeil) ===")
v2 = art2b["verdict_critere_2"]
print(f"marginales appariees : {v2['marginales_appariees']}")
for nom, b in v2["bras"].items():
    print(f"  {nom:18s} : {'SATISFAIT' if b['satisfait'] else 'non satisfait'}  "
          f"(gap negatif : {b['gap_negatif']}, hors bande : {b['hors_bande']}, "
          f"gaps : {[g['additivity_gap'] for g in b['gaps']]})")
MESURE_ETAT_COUPLE_V3 = v2


=== 1. NECESSITE : le token du parent explique-t-il la cible ? ===
n = 20000 pas, graine de donnees 1, decalage 1
regime             R2(token->cible)   var(cible)  I(tok;cible)   I(tok;obs)
etat_couple                 +1.0000     0.666662       +1.0986      +0.0096
emission_variante           -0.0002     0.504897       +0.0001      +0.0000
independant                 -0.0114     0.041766       +0.0002      +0.0001
appariement des marginales : exact = True  ecart theorique max 0.00e+00  ecart empirique max 0.0052

=== 2. ENTRAINEMENT : le gap sort-il de la bande du temoin ? ===
decalage = 1  1200 pas  graines modele = [0, 1, 7]  temoin = 6 initialisations
independance MESUREE des facteurs : conditionnel 0.009240 nats / independant 0.000153 nats
[conditionnel_etat]  bande gap [+0.000, +0.000]  energie [0.071, 0.156]  R2(token->cible) = +1.0000
  seed 0: val=2.1909 gate=PASS  gap=+0.000 [dans_bande]  energie=0.034 [sous_bande]  ovl_principal=0.260  dims=(2,2,4)
  seed 1: val=2.1905 gate=P

### Lecture du régime 2b (dépendance rendue nécessaire) -- troisième passe

- la **nécessité est mesurée avant tout entraînement** : sur le banc à état couplé, `R^2(token -> cible) = +1.0000` et l'information mutuelle `I(token ; belief) = 1.0986 nats` -- le plafond `log 3`. La cible de l'enfant n'a pas d'autre route que le token observé du parent. Sur les bancs où la dépendance n'est pas nécessaire, la même mesure rend ~0 : variante d'émission `R^2 = -0.0002`, contrôle indépendant `R^2 = -0.0114`. Les marginales restent appariées (écart théorique 0.0, empirique 0.0052 à 20 000 pas) ;
- l'**entraînement** passe le gate aux trois graines, et les pertes se serrent contre le contrôle : 2.1909 / 2.1905 / 2.1898 contre 2.1897 / 2.1900 / 2.1881, toutes sous l'uniforme joint 2 ln 3 = 2.1972 ;
- **le verdict du critère 2 est rendu par le code, et il est négatif** : gap `+0.000` aux trois graines dans le bras « état couplé », gap jamais négatif, jamais hors bande -- **non satisfait**, témoin à `+0.000` sur ses six initialisations.

**Le résultat est rapporté tel quel, comme le pré-enregistrement le prévoyait.** Le protocole avait fixé d'avance ce cas : « si le gap reste nul dans le bras état couplé, le résultat est rapporté tel quel ». Le gap est resté nul : sur un banc où la dépendance est pourtant **nécessaire** (`R^2 = 1`) et où le modèle apprend (gate PASS aux trois graines), l'`additivity_gap` -- entier au rang près -- ne quitte pas zéro. Ce que cela établit : l'instrument décompte des **unités de rang**, et ce banc n'en met pas en jeu ; la signature géométrique recherchée (recouvrement accru des sous-espaces de lecture, donc gap négatif) n'est pas détectable par cette métrique ici, quel que soit le budget d'entraînement. Ce que cela n'établit pas : que la signature n'existe pas. La métrique continue `overlap_energy` rend 0.034 / 0.052 / 0.078 contre une bande de témoin `[0.071, 0.156]` -- deux graines sur trois **sous** la bande, un recouvrement plus petit que celui des initialisations non entraînées, donc un signe cohérent mais **inverse** de celui que le critère demandait (il demandait plus de recouvrement, pas moins). L'issue honnête du suivi #20071 sur ce point : l'`additivity_gap` n'est pas l'instrument de ce phénomène, et le critère 2 reste ouvert dans les deux sens.

## Exercices


### Exercice 1 — Sweep d'overlap : où l'additivité se dégrade-t-elle ?

Les trois régimes ci-dessus sont mesurés à **paramètres fixes** : overlap 0 ou 0,4, quatre facteurs, 2 048 tokens. Les trois exercices suivants font varier chacun **un** paramètre du protocole et mesurent ce que la fenêtre fixe ne peut pas voir. Chaque énoncé fixe les paramètres exacts à employer : le résultat attendu est vérifié par une exécution de référence (hors notebook).

Les primitives importées plus haut (`make_factor_bases`, `synthesize_activations`, `weighted_pca`, `nc_at`, `basis_overlap`) suffisent — ne rien réécrire du module.

Le carnet compare **deux** points d'overlap : 0 (Exemple guidé 1) et 0,4 (Exemple guidé 2). Un point ne fait pas une courbe : on ne sait pas si la dégradation est progressive, brutale, ni si sa dispersion entre graines change.

**Protocole** — mode `superposed`, quatre facteurs, 2 048 tokens, les cinq graines `RNG_SEEDS`, pour `overlap` ∈ {0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6} :

1. mesurer l'écart d'additivité `|NC@95 joint − Σ NC@95 factoriels|` pour chaque graine ;
2. tabuler, par overlap, l'écart **moyen**, **min** et **max** ;
3. qualifier : la moyenne croît-elle de façon monotone ? La dispersion entre graines se comporte-t-elle comme la moyenne ?

**Indices**

- Recopier le motif de l'Exemple guidé 2 en paramétrant `overlap` (`make_factor_bases(..., mode="superposed", overlap=..., rng=...)`).
- Le « min » et le « max » sur les graines sont deux nombres distincts de la moyenne : c'est la dispersion qui porte le fait marquant.

**Vocabulaire de l'axe x** — le paramètre de grille `overlap` est l'**écart-type des coefficients de mélange** injectés entre sous-espaces (non borné), pas l'overlap géométrique lui-même. Pour un axe homogène au vocabulaire du carnet, reporter aussi l'overlap **mesuré** par point de grille : `basis_overlap(bases[i], bases[j]).max()` sur les paires, comme `max_pair_overlap` l'imprime dans les Exemples guidés.

**Ce que l'exercice fait mesurer** — la dégradation de l'additivité avec le paramètre de mélange *et* la stabilité de sa mesure.


In [15]:
# Exercice 1 — Sweep d'overlap : ou l'additivite se degrade-t-elle ? (a completer)
# TODO etudiant : mesurer l'ecart d'additivite sur une grille d'overlap.
# Etape 1 : OVERLAPS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6) ; quatre facteurs, 2048 tokens, RNG_SEEDS.
# Etape 2 : pour chaque overlap, mesurer l'ecart par graine (motif de l'Exemple guide 2).
# Etape 3 : tabuler ecart moyen, min, max par overlap et qualifier la monotonie de la moyenne.
# Indice : la dispersion entre graines (min / max) est le second fait a lire, pas seulement la moyenne.
# Indice : reporter aussi, par point de grille, l'overlap geometrique mesure (basis_overlap(bases[i], bases[j]).max())
#         pour que l'axe x se lise dans le vocabulaire du carnet (le parametre overlap est un ecart-type de melange).

OVERLAPS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6)

def ecart_additivite(n_factors, n_tokens, overlap, seed):
    """Ecart |NC@95 joint - somme des NC@95 factoriels| pour un tirage donne."""
    # TODO etudiant
    pass

# for ov in OVERLAPS: ...   # decommenter une fois implemente
print("Exercice 1 a completer : sweep d'overlap et stabilite de l'ecart d'additivite.")


Exercice 1 a completer : sweep d'overlap et stabilite de l'ecart d'additivite.


### Exercice 2 — Nombre de facteurs : le côté joint sature-t-il ?

Le carnet fixe `N_FACTORS = 4`. Ce nombre pilote la taille de chaque sous-espace (`dim // n_factors` : 128, 64 ou 32 sur les 256 dimensions du stream). Que devient l'écart d'additivité quand le **nombre** de facteurs change, à budget de dimensions constant ?

**Protocole** — mode `orthogonal`, 2 048 tokens, les cinq graines `RNG_SEEDS`, pour `n_factors` ∈ {2, 4, 8} :

1. mesurer `NC@95` joint et la somme des `NC@95` factoriels (moyennes sur les graines) ;
2. tabuler aussi `dim_per_factor` (`make_factor_bases` le renvoie) ;
3. qualifier : lequel des deux côtés — joint ou somme — bouge avec le nombre de facteurs, et lequel sature ?

**Indices**

- Le côté joint sature à une valeur proche du **budget total** de dimensions ; le côté somme, lui, dépend du découpage.
- À huit facteurs, `dim_per_factor` vaut 32 : vérifier que chaque projection factorielle a encore assez de dimensions pour porter son `NC@95`.

**Ce que l'exercice fait mesurer** — l'additivité n'est pas symétrique entre ses deux côtés, et la saturation du côté joint est une propriété du budget de dimensions.


In [16]:
# Exercice 2 — Nombre de facteurs : le cote joint sature-t-il ? (a completer)
# TODO etudiant : mesurer les deux cotes de l'additivite en fonction de n_factors.
# Etape 1 : N_FACTORS_GRID = (2, 4, 8) ; mode orthogonal, 2048 tokens, RNG_SEEDS.
# Etape 2 : pour chaque n_factors, moyenne sur graines de NC@95 joint, de la somme des NC@95
#           factoriels et de l'ecart ; recuperer dim_per_factor renvoye par make_factor_bases.
# Etape 3 : tabuler (n_factors, dim_per_factor, joint, somme, ecart) et dire quel cote sature.
# Indice : le cote joint est borne par le budget total des 256 dimensions du stream.

N_FACTORS_GRID = (2, 4, 8)

def deux_cotes_additivite(n_factors, n_tokens, seed):
    """Retourne (joint, somme_factorielle, dim_per_factor) pour un tirage donne."""
    # TODO etudiant
    pass

# for nf in N_FACTORS_GRID: ...   # decommenter une fois implemente
print("Exercice 2 a completer : nombre de facteurs et saturation du cote joint.")


Exercice 2 a completer : nombre de facteurs et saturation du cote joint.


### Exercice 3 — Nombre de tokens : le plancher d'échantillonnage

Le carnet fixe `N_TOKENS = 2 048` et traite l'écart d'additivité comme une propriété **géométrique**. Or une PCA sur $n$ échantillons ne peut pas rendre plus de $n$ directions (rang de la matrice) : le côté joint a donc un **plancher** qui n'a rien de géométrique. Le carnet a déjà rencontré ce genre d'artefact avec l'hypothèse nulle (`null_overlap_distribution`) — cet exercice le mesure pour la dimension.

**Protocole** — mode `orthogonal`, quatre facteurs, les cinq graines `RNG_SEEDS`, pour `n_tokens` ∈ {64, 256, 1 024, 4 096} :

1. mesurer `NC@95` joint et la somme des `NC@95` factoriels (moyennes sur les graines) ;
2. tabuler l'écart d'additivité correspondant ;
3. vérifier le plancher au plus petit échantillon, puis dire ce qui reste de l'écart à 4 096 tokens.

**Indices**

- À 64 tokens, le côté joint est borné par le rang de la matrice centrée (≤ 63) : le comparer à sa borne.
- Le côté somme, lui, est mesuré sur des projections de dimension 64 : il ne subit pas la même borne.
- La conclusion utile : l'écart à petit échantillon mesure surtout le **manque de données**, pas la géométrie.

**Ce que l'exercice fait mesurer** — le plancher d'échantillonnage de `NC@95` et la part d'écart qui survit à grand échantillon.


In [17]:
# Exercice 3 — Nombre de tokens : le plancher d'echantillonnage (a completer)
# TODO etudiant : mesurer les deux cotes de l'additivite en fonction de n_tokens.
# Etape 1 : N_TOKENS_GRID = (64, 256, 1024, 4096) ; mode orthogonal, 4 facteurs, RNG_SEEDS.
# Etape 2 : pour chaque n_tokens, moyenne sur graines de NC@95 joint, de la somme des NC@95
#           factoriels et de l'ecart.
# Etape 3 : comparer le joint a 64 tokens a sa borne de rang, puis lire ce qui reste a 4096.
# Indice : une PCA sur n echantillons ne rend pas plus de n directions -- borner le joint par n.

N_TOKENS_GRID = (64, 256, 1024, 4096)

def plancher_echantillonnage(n_tokens, n_factors, seed):
    """Retourne (joint, somme_factorielle) pour un tirage donne."""
    # TODO etudiant
    pass

# for nt in N_TOKENS_GRID: ...   # decommenter une fois implemente
print("Exercice 3 a completer : plancher d'echantillonnage de NC@95.")


Exercice 3 a completer : plancher d'echantillonnage de NC@95.


In [18]:
# Protocole freeze-and-vary : on gele UN facteur et on varie l'AUTRE
# (cle d'isolement de la contribution factorielle, cf. arXiv:2602.02385)
# Implementation : pour chaque seed, on genere 2 bases orthogonales via
# make_factor_bases(2, 16) ; on gele bases[0] et on regenere bases[1] a chaque
# realisation. NC@p sur les activations devrait refleter 2 dimensions dominantes
# stables (gele) + une sensibilite au facteur varie.
# NB : on ne boucle PAS sur n_factors en externe -- la complexite du protocole
# est dans la variation du second facteur, pas dans le nombre de facteurs.

FREEZE_AND_VARY_SEEDS = [0, 1, 7, 42, 99]  # 5 seeds conformes a l'AC >=4

frozen_nc95_per_seed = []
varied_nc95_per_seed = []
for seed in FREEZE_AND_VARY_SEEDS:
    rng = np.random.default_rng(seed)
    # Bases orthogonales gelees : meme base pour toutes les realisations du facteur varie
    bases, _ = make_factor_bases(2, dim=16, mode="orthogonal", rng=rng)
    frozen_factor = bases[0]  # (16, 8)
    nc95_real = []
    for r in range(20):
        rng_r = np.random.default_rng(seed * 1000 + r)
        # Nouveau facteur varie : meme dim_per_factor (8) que le gele
        # NB : make_factor_bases retourne (bases, dim_per_factor), donc [0][1] = bases[1]
        varied_factor = make_factor_bases(2, dim=16, mode="orthogonal", rng=rng_r)[0][1]
        # 256 tokens, chacun = projection sur gele + projection sur varie + bruit
        n_tok = 256
        z_frozen = rng_r.standard_normal((n_tok, 8))  # coefficients gele
        z_varied = rng_r.standard_normal((n_tok, 8))  # coefficients varie
        # Activations : (n_tok, dim=16) via (n_tok, 8) @ (8, 16) = (n_tok, 16)
        X = z_frozen @ frozen_factor.T + z_varied @ varied_factor.T + 0.1 * rng_r.standard_normal((n_tok, 16))
        _, _, _, evr = weighted_pca(X)
        nc_dict = nc_at(evr, thresholds=(95,))
        nc95_real.append(nc_dict[95])
    frozen_nc95_per_seed.append(np.mean(nc95_real))
    varied_nc95_per_seed.append(np.std(nc95_real))

freeze_vary_results = {
    "frozen_nc95_mean": float(np.mean(frozen_nc95_per_seed)),
    "frozen_nc95_std": float(np.std(frozen_nc95_per_seed)),
    "varied_sensitivity_mean": float(np.mean(varied_nc95_per_seed)),
}

print("=== Freeze-and-vary : stabilite du sous-espace gele + sensibilite au varie (5 seeds) ===")
print(f"  frozen NC@95 = {freeze_vary_results['frozen_nc95_mean']:.3f} +/- {freeze_vary_results['frozen_nc95_std']:.3f}")
print(f"  varied sensitivity (ecart-type inter-realisations) = {freeze_vary_results['varied_sensitivity_mean']:.3f}")
print()
print("Verdict attendu : frozen NC@95 doit etre proche du nombre de facteurs dominants (2 ici, gele + varie) ; varied sensitivity reflete la variabilite du facteur varie.")


=== Freeze-and-vary : stabilite du sous-espace gele + sensibilite au varie (5 seeds) ===
  frozen NC@95 = 11.490 +/- 0.185
  varied sensitivity (ecart-type inter-realisations) = 0.593

Verdict attendu : frozen NC@95 doit etre proche du nombre de facteurs dominants (2 ici, gele + varie) ; varied sensitivity reflete la variabilite du facteur varie.


## Verdict multi-seed et hypothese falsifiable

Les hypotheses du grain #15478 — H1 a H3 a la livraison initiale (#15514), H4 ajoutee
par le suivi du 15/09 (criteres « courbe en k » et « axes de graine independants »,
section dediee ci-dessous) :

1. **H1 (separation)** : les sous-espaces factoriels reels sont PLUS orthogonaux que
   des paires de sous-espaces aleatoires apparies. Le sens implemente depuis la
   livraison initiale est `max_real < null_q95` : l'overlap reel passe SOUS le
   quantile 95 de la distribution nulle (cf. Exemple guidé 3, « sep? = overlap < null95 »).
   Une redaction anterieure de cette ligne affichait `>` a tort : coquille de prose,
   le code a toujours teste `<`.
2. **H2 (additivite jointe)** : la dimension jointe est compatible avec une organisation additive dans le regime factorise orthogonal (test : gap additivite faible et stable sur les seeds).
3. **H3 (sensibilite au bruit)** : la structure survit a un niveau de bruit pre-enregistre (0.1), mais disparait au-dela (0.5).
4. **H4 (courbe overlap-vs-k, suivi du 15/09)** : tracee en fonction de k plutot que
   reduite a un scalaire a k fixe, l'overlap franchit la bande nulle exactement au
   `k_cross = n_privees + 1` pre-enregistre dans le regime superpose, et reste sous la
   bande a tout k dans le regime orthogonal — data-seed et null-seed sur deux axes
   d'entropie disjointe (section H4 ci-dessous).


In [19]:
# H1 : separation des sous-espaces factoriels vs hypothese nulle
# Multi-seed 5 seeds conformement a l'AC #15478 (>= 4 seeds avec IC)
# On utilise la metrique corrigee d'Exemple guide 3 (results_noise_sweep[0.0],
# regime orthogonal = bruit nul) qui projette sur les sous-espaces factoriels
# reels et compare a des paires de sous-espaces aleatoires apparies.
# NB : le test archaique null_overlap_distribution(Q aleatoire vs full space)
#      donnait null95=1.0 systematiquement (artefact reconnu dans la cellule 14).

VERDICT_SEEDS = [0, 1, 7, 42, 99]

h1_per_seed = {}
for seed in VERDICT_SEEDS:
    r = results_noise_sweep[0.0].get(seed, None)
    if r is None:
        # Fallback : recalculer pour ce seed
        rng = np.random.default_rng(seed)
        bases, dim_per = make_factor_bases(N_FACTORS, N_DIM_RESIDUAL, mode="orthogonal", rng=rng)
        X, w = synthesize_activations(bases, N_TOKENS, signal_var=1.0, noise_var=0.0, rng=rng)
        # Calcul direct via projection
        max_real = 0.0
        for i in range(N_FACTORS):
            for j in range(i + 1, N_FACTORS):
                proj_i = X @ bases[i] @ bases[i].T
                proj_j = X @ bases[j] @ bases[j].T
                _, comps_i, _, _ = weighted_pca(proj_i, w)
                _, comps_j, _, _ = weighted_pca(proj_j, w)
                B_i = comps_i[:, :dim_per]
                B_j = comps_j[:, :dim_per]
                M = basis_overlap(B_i, B_j)
                if M.max() > max_real:
                    max_real = float(M.max())
        # Null : paires aleatoires appariees
        null_pairs = []
        rng_null = np.random.default_rng(seed * 31 + 17)
        for _ in range(50):
            A = rng_null.standard_normal((N_DIM_RESIDUAL, dim_per))
            Q1, _ = np.linalg.qr(A)
            A = rng_null.standard_normal((N_DIM_RESIDUAL, dim_per))
            Q2, _ = np.linalg.qr(A)
            M = basis_overlap(Q1, Q2)
            null_pairs.append(float(M.max()))
        null_q95 = float(np.quantile(np.array(null_pairs), 0.95))
        h1_per_seed[seed] = {
            "real": max_real,
            "null95": null_q95,
            "separates": max_real < null_q95,
        }
    else:
        h1_per_seed[seed] = {
            "real": r["max_pair_overlap"],
            "null95": r["null_q95_overlap"],
            "separates": r["separates_from_null"],
        }

print("=== Verdict multi-seed H1 (5 seeds) : separation des sous-espaces factoriels ===")
print("Test : overlap reel (sur projections) < q95 distribution nulle (paires aleatoires appariees)")
for seed, v in h1_per_seed.items():
    verdict = "SEPARATES" if v["separates"] else "MERGES-WITH-NULL"
    print(f"  seed={seed}: real_overlap={v['real']:.4f}, null95={v['null95']:.4f}, {verdict}")

real_vals = [v["real"] for v in h1_per_seed.values()]
null_vals = [v["null95"] for v in h1_per_seed.values()]
n_sep = sum(1 for v in h1_per_seed.values() if v["separates"])

print()
print(f"Moyenne real_overlap = {np.mean(real_vals):.4f} +/- {np.std(real_vals):.4f}")
print(f"Moyenne null95      = {np.mean(null_vals):.4f} +/- {np.std(null_vals):.4f}")
print(f"Seeds separating from null : {n_sep} / {len(VERDICT_SEEDS)}")
print()
if n_sep == len(VERDICT_SEEDS):
    print("H1 SUPPORTED cross-seed : tous les 5 seeds separent les sous-espaces mieux que l'aleatoire apparie.")
elif n_sep >= 3:
    print("H1 PARTIALLY SUPPORTED : majorite des seeds separent, mais resultats heterogenes.")
else:
    print("H1 INCONCLUSIVE : la separation n'est pas reproductible cross-seed.")


=== Verdict multi-seed H1 (5 seeds) : separation des sous-espaces factoriels ===
Test : overlap reel (sur projections) < q95 distribution nulle (paires aleatoires appariees)
  seed=0: real_overlap=0.0000, null95=0.2609, SEPARATES
  seed=1: real_overlap=0.0000, null95=0.2662, SEPARATES
  seed=7: real_overlap=0.0000, null95=0.2663, SEPARATES
  seed=42: real_overlap=0.0000, null95=0.2775, SEPARATES
  seed=99: real_overlap=0.0000, null95=0.2628, SEPARATES

Moyenne real_overlap = 0.0000 +/- 0.0000
Moyenne null95      = 0.2667 +/- 0.0057
Seeds separating from null : 5 / 5

H1 SUPPORTED cross-seed : tous les 5 seeds separent les sous-espaces mieux que l'aleatoire apparie.


## Hypothese H4 — Courbe overlap-vs-k et axes de graine independants (suivi du 15/09)

Le verdict H1 ci-dessus resume la separation par un **scalaire a k fixe** (`k = dim_per_factor`). Le commentaire du 15/09 sur #15478 demande l'overlap **rapporte comme courbe en k** — *« un scalaire depend du choix de k, la courbe dit ou la separation commence et ou elle sature »* — et la separation des axes de graine (data-seed vs graine du temoin). H4 porte ces deux exigences au regime synthetique de ce notebook.

**Pre-enregistrement** — ecrit AVANT la cellule de mesure ci-dessous :

- **Construction** : deux facteurs dans R^256, `dim_per = 12` directions chacun. Chaque direction privee `m` porte une variance `12 - m` (ordre spectral strict, 12, 11, ...) ; `n_shared = 5` directions basse-variance (variance 2 **par facteur**) sont communes aux deux facteurs. Dans chaque projection oracle, une direction partagee ne porte que `2 + 2 = 4 < 6` de variance : l'ordre PCA place donc **toutes** les directions privees avant les partagees.
- **P1 (regime orthogonal, n_shared = 0)** : la courbe reelle `overlap(k)` reste sous la bande nulle (quantile 95 des paires aleatoires appariees) pour **tout** k de 1 a 12.
- **P2 (regime superpose, n_shared = 5)** : la courbe reste sous la bande pour `k <= 7`, franchit la bande **exactement a `k_cross = n_privees + 1 = 8`** (premiere direction partagee admise dans le top-k) et reste au-dessus pour tout `k >= 8`.
- **Regle de decision (pre-enregistree)** : H4 SUPPORTED si P1 et P2 tiennent **par seed** sur au moins 4 des 5 data-seeds, croisement observe egal a 8 exactement (aucune tolerance) ; sinon REFUTED avec diagnostic par seed.
- **Axes independants** : la graine de donnees (`default_rng([1000, data_seed])`, data_seed dans {0, 1, 7, 42, 99}) et la graine du temoin nul (`default_rng([2000, null_seed])`, 5 null-seeds x 40 tirages = 200 par k) derivent de **deux entropies disjointes** — la collision du type `(seed=0, noise=1.0) = (seed=1, noise=0.0)` qui affectait le sweep de bruit (corrigee dans cette meme livraison : multiplicateur 100 -> 1000) ne peut plus se produire.
- **Limitation explicite** : le temoin nul de H4 reste le null de **sous-espaces aleatoires apparies** (meme modele nul que H1). Le critere du 15/09 « temoin nul sur >= 5 initialisations non entrainees du meme modele » exige un banc d'entrainement : il n'est ni couvert ni revendique ici (cf. Limites et bilan honnete).


In [20]:
# H4 — pre-enregistrement : cette cellule FIXE la prediction AVANT toute mesure.
# La cellule de mesure suivante lit H4_PREDICTIONS et ne redefinit rien : le verdict
# est evalue contre ce qui est imprime ici, pas contre un critere repere apres coup.
H4_DIM = 256                    # dimension ambiante (proxy residual stream)
H4_DIM_PER = 12                 # directions par facteur
H4_N_SHARED = 5                 # directions partagees (regime superpose)
H4_N_TOKENS = 2048
H4_DATA_SEEDS = (0, 1, 7, 42, 99)   # >= 5 data-seeds (AC multi-seed)
H4_NULL_SEEDS = (0, 1, 7, 42, 99)   # axe INDEPENDANT : entropies [2000, ...] disjointes de [1000, ...]
H4_NULL_DRAWS_PER_SEED = 40         # 5 x 40 = 200 tirages nuls par k
H4_K_GRID = tuple(range(1, H4_DIM_PER + 1))
H4_VAR_SHARED_PER_FACTOR = 2.0      # var partagee par facteur -> 2+2 = 4 < 6 en projection

H4_N_PRIVATE = H4_DIM_PER - H4_N_SHARED
H4_K_CROSS_PREDICTED = H4_N_PRIVATE + 1   # premiere direction partagee dans le top-k

H4_PREDICTIONS = {
    "p1_orthogonal": "courbe reelle sous la bande nulle (q95) pour TOUT k de 1 a 12",
    "p2_superposed": "courbe sous la bande pour k <= 7, au-dessus pour tout k >= 8",
    "k_cross_predicted": H4_K_CROSS_PREDICTED,
    "decision_rule": ("SUPPORTED si P1 et P2 tiennent par seed sur >= 4/5 data-seeds, "
                      "croisement observe == 8 exactement"),
    "null_model": "paires de sous-espaces aleatoires apparies (QR), 200 tirages/k",
}
print("=== H4 : pre-enregistrement (AVANT mesure) ===")
for key, val in H4_PREDICTIONS.items():
    print(f"  {key} : {val}")
print(f"  construction : var privee m -> {H4_DIM_PER} - m ; var partagee -> {H4_VAR_SHARED_PER_FACTOR} par facteur")
print(f"  consequence analytique : k_cross = n_privees + 1 = {H4_N_PRIVATE} + 1 = {H4_K_CROSS_PREDICTED}")


=== H4 : pre-enregistrement (AVANT mesure) ===
  p1_orthogonal : courbe reelle sous la bande nulle (q95) pour TOUT k de 1 a 12
  p2_superposed : courbe sous la bande pour k <= 7, au-dessus pour tout k >= 8
  k_cross_predicted : 8
  decision_rule : SUPPORTED si P1 et P2 tiennent par seed sur >= 4/5 data-seeds, croisement observe == 8 exactement
  null_model : paires de sous-espaces aleatoires apparies (QR), 200 tirages/k
  construction : var privee m -> 12 - m ; var partagee -> 2.0 par facteur
  consequence analytique : k_cross = n_privees + 1 = 7 + 1 = 8


In [21]:
# H4 — mesure : courbe overlap-vs-k contre bande nulle appariee, par data-seed.
# data-seed : default_rng([1000, ds]) ; null-seed : default_rng([2000, ns]) —
# deux SeedSequence d'entropies disjointes, aucune collision possible entre axes.
def h4_build_factors(rng_data, n_shared):
    """Deux sous-espaces de R^H4_DIM : blocs prives disjoints + bloc partage."""
    Q, _ = np.linalg.qr(rng_data.standard_normal((H4_DIM, H4_DIM)))
    n_priv = H4_DIM_PER - n_shared
    cols_priv1 = list(range(n_priv))                 # directions prives facteur 1
    cols_priv2 = list(range(n_priv, 2 * n_priv))     # directions prives facteur 2
    cols_shared = list(range(100, 100 + n_shared))   # bloc partage basse variance
    B1 = Q[:, cols_priv1 + cols_shared]
    B2 = Q[:, cols_priv2 + cols_shared]
    var_priv = [float(H4_DIM_PER - m) for m in range(n_priv)]
    var = np.array(var_priv + [H4_VAR_SHARED_PER_FACTOR] * n_shared)
    return B1, B2, var

def h4_real_curve(rng_data, n_shared):
    """overlap(k) = max |cos| entre top-k composantes PCA des projections oracle."""
    B1, B2, var = h4_build_factors(rng_data, n_shared)
    z1 = rng_data.standard_normal((H4_N_TOKENS, H4_DIM_PER)) * np.sqrt(var)[None, :]
    z2 = rng_data.standard_normal((H4_N_TOKENS, H4_DIM_PER)) * np.sqrt(var)[None, :]
    X = z1 @ B1.T + z2 @ B2.T
    _, comps1, _, _ = weighted_pca(X @ B1 @ B1.T)
    _, comps2, _, _ = weighted_pca(X @ B2 @ B2.T)
    return [float(basis_overlap(comps1[:, :k], comps2[:, :k]).max()) for k in H4_K_GRID]

# Bande nulle appariee : par k, 5 null-seeds x 40 tirages, meme metrique max|cos|.
h4_null_band = {}
for k in H4_K_GRID:
    draws = []
    for null_seed in H4_NULL_SEEDS:
        rng_null = np.random.default_rng([2000, null_seed])
        for _ in range(H4_NULL_DRAWS_PER_SEED):
            Q1, _ = np.linalg.qr(rng_null.standard_normal((H4_DIM, k)))
            Q2, _ = np.linalg.qr(rng_null.standard_normal((H4_DIM, k)))
            draws.append(float(basis_overlap(Q1, Q2).max()))
    q05, q50, q95 = np.quantile(np.array(draws), (0.05, 0.50, 0.95))
    h4_null_band[k] = (float(q05), float(q50), float(q95))

h4_results = {}
for regime, n_shared in (("orthogonal", 0), ("superposed", H4_N_SHARED)):
    curves = np.array([
        h4_real_curve(np.random.default_rng([1000, ds]), n_shared)
        for ds in H4_DATA_SEEDS
    ])
    band_q95 = np.array([h4_null_band[k][2] for k in H4_K_GRID])
    below = curves < band_q95                              # (seed, k) contre q95
    k_cross_obs = {}
    for s_idx, ds in enumerate(H4_DATA_SEEDS):
        above = [k for i, k in enumerate(H4_K_GRID) if not below[s_idx, i]]
        k_cross_obs[ds] = above[0] if above else None
    h4_results[regime] = {"curves": curves, "below": below, "k_cross_obs": k_cross_obs}
    print(f"=== H4 regime {regime} (n_shared={n_shared}) : overlap(k), {len(H4_DATA_SEEDS)} data-seeds ===")
    print(f"{'k':>3} {'real_mean':>10} {'real_std':>9} {'null_q05':>9} {'null_q50':>9} {'null_q95':>9} {'<q95':>6}")
    for i, k in enumerate(H4_K_GRID):
        q05, q50, q95 = h4_null_band[k]
        n_below = int(below[:, i].sum())
        print(f"{k:>3d} {curves[:, i].mean():>10.4f} {curves[:, i].std():>9.4f} "
              f"{q05:>9.4f} {q50:>9.4f} {q95:>9.4f} {n_below:>4d}/5")
    print(f"  croisements observes (premier k au-dessus de la bande) : {k_cross_obs}")
    print()

# Verdict contre la regle PRE-ENREGISTREE (aucun critere redefini ici)
k_pred = H4_PREDICTIONS["k_cross_predicted"]
p1_ok_seeds = sum(
    all(h4_results["orthogonal"]["below"][s, :]) for s in range(len(H4_DATA_SEEDS))
)
p2_ok_seeds = sum(
    all(h4_results["superposed"]["below"][s, :k_pred - 1])
    and all(~h4_results["superposed"]["below"][s, k_pred - 1:])
    for s in range(len(H4_DATA_SEEDS))
)
print("=== H4 : verdict contre la regle pre-enregistree ===")
print(f"P1 (orthogonal sous la bande a tout k) : {p1_ok_seeds}/5 data-seeds")
print(f"P2 (croisement exact a k={k_pred}) : {p2_ok_seeds}/5 data-seeds")
if p1_ok_seeds >= 4 and p2_ok_seeds >= 4:
    print(f"H4 SUPPORTED : courbe overlap-vs-k discriminante, croisement a k={k_pred} comme predit.")
else:
    print("H4 REFUTED : la regle pre-enregistree n'est pas tenue (diagnostic par seed ci-dessus).")


=== H4 regime orthogonal (n_shared=0) : overlap(k), 5 data-seeds ===
  k  real_mean  real_std  null_q05  null_q50  null_q95   <q95
  1     0.0000    0.0000    0.0022    0.0460    0.1148    5/5
  2     0.0000    0.0000    0.0407    0.0906    0.1478    5/5
  3     0.0000    0.0000    0.0658    0.1131    0.1803    5/5
  4     0.0000    0.0000    0.0846    0.1288    0.1821    5/5
  5     0.0000    0.0000    0.0996    0.1341    0.1819    5/5
  6     0.0000    0.0000    0.1102    0.1444    0.1922    5/5
  7     0.0000    0.0000    0.1188    0.1519    0.1984    5/5
  8     0.0000    0.0000    0.1242    0.1563    0.1972    5/5
  9     0.0000    0.0000    0.1283    0.1631    0.2221    5/5
 10     0.0000    0.0000    0.1386    0.1687    0.2071    5/5
 11     0.0000    0.0000    0.1431    0.1743    0.2217    5/5
 12     0.0000    0.0000    0.1459    0.1786    0.2192    5/5
  croisements observes (premier k au-dessus de la bande) : {0: None, 1: None, 7: None, 42: None, 99: None}



=== H4 regime superposed (n_shared=5) : overlap(k), 5 data-seeds ===
  k  real_mean  real_std  null_q05  null_q50  null_q95   <q95
  1     0.0006    0.0005    0.0022    0.0460    0.1148    5/5
  2     0.0015    0.0006    0.0407    0.0906    0.1478    5/5
  3     0.0020    0.0008    0.0658    0.1131    0.1803    5/5
  4     0.0028    0.0008    0.0846    0.1288    0.1821    5/5
  5     0.0031    0.0009    0.0996    0.1341    0.1819    5/5
  6     0.0046    0.0015    0.1102    0.1444    0.1922    5/5
  7     0.0060    0.0017    0.1188    0.1519    0.1984    5/5
  8     0.9759    0.0255    0.1242    0.1563    0.1972    0/5
  9     0.9759    0.0255    0.1283    0.1631    0.2221    0/5
 10     0.9886    0.0050    0.1386    0.1687    0.2071    0/5
 11     0.9889    0.0051    0.1431    0.1743    0.2217    0/5
 12     0.9917    0.0053    0.1459    0.1786    0.2192    0/5
  croisements observes (premier k au-dessus de la bande) : {0: 8, 1: 8, 7: 8, 42: 8, 99: 8}

=== H4 : verdict contre la regle

### Interpretation : lecture de la courbe H4

**Sortie obtenue** : deux familles de courbes `overlap(k)` (5 data-seeds chacune, moyenne +/- std) contre une bande nulle appariee (q05/q50/q95, 200 tirages par k).

| Aspect | Valeur | Signification |
|---|---|---|
| Regime orthogonal | courbe ~0 a tout k, sous la bande | separation totale, independante du choix de k |
| Superpose, k <= 7 | courbe sous la bande | le top-k ne contient que des directions privees : un scalaire calcule ici dirait « faible recouvrement » |
| Superpose, k >= 8 | courbe ~0.98, au-dessus de la bande | les directions partagees entrent dans le top-k : un scalaire calcule ici dirait « fort recouvrement » |
| Bande nulle | q50 monte de ~0.05 (k=1) a ~0.18 (k=12) | le seuil de recouvrement aleatoire depend de k — pourquoi un scalaire a k fixe est fragile |

**Points cles** :

1. La lecture du recouvrement a k fixe **s'inverse** selon le k choisi (faible pour k <= 7, fort pour k >= 8 dans le regime superpose) : seule la courbe dit ou les directions partagees entrent dans le spectre et ou leur recouvrement sature. Un recouvrement au-dessus du quantile 95 du null n'est **pas** un recouvrement « semblable au null » : il le depasse. Inversement, le test de faible recouvrement contre le hasard serait mieux calibre sur la queue inferieure (q05) ; le seuil q95 ici est un discriminant descriptif pre-enregistre, pas une p-value de separation.
2. Le croisement observe a `k = n_privees + 1 = 8` confirme la prediction pre-enregistree sur ce generateur : la geometrie mesuree reflete les directions partagees placees en fin de spectre. Il ne constitue pas une preuve sur des activations de modele entraine.
3. Data-seed et null-seed derivent de familles de graines distinctes (`[1000, ...]` vs `[2000, ...]`) : la variabilite rapportee (std entre data-seeds et percentiles des paires aleatoires) ne provient plus de la collision de graines entre conditions du sweep de bruit.

> **Note technique** : le null de H4 reste un null de **sous-espaces aleatoires apparies** (meme modele nul que H1). Le temoin « >= 5 initialisations non entrainees » du 15/09 exige un banc de modele : hors scope de ce notebook synthetique, non revendique (cf. Limites).


## Visualisations et limites

Cette section prepare des matrices d'overlap pour chaque regime et un resume textuel des resultats. Une variante matplotlib est documentee en exercice optionnel ; les chiffres exacts sont la sortie de reference.


In [22]:
# Matrice d'overlap moyenne pour le regime orthogonal (seed=0, demonstratif)
rng = np.random.default_rng(0)
bases, _ = make_factor_bases(N_FACTORS, N_DIM_RESIDUAL, mode="orthogonal", rng=rng)
overlap_matrix = np.zeros((N_FACTORS, N_FACTORS))
for i in range(N_FACTORS):
    for j in range(N_FACTORS):
        if i == j:
            overlap_matrix[i, j] = 1.0
        elif i < j:
            M = basis_overlap(bases[i], bases[j])
            overlap_matrix[i, j] = overlap_matrix[j, i] = float(M.max())
print("Matrice d'overlap (regime orthogonal, seed=0) :")
for row in overlap_matrix:
    print("  [" + "  ".join(f"{v:.3f}" for v in row) + "]")
print()
print("Interpretation : diagonale = 1.0 (auto-overlap), hors-diagonale proche de 0 = orthogonalite,")
print("proche de 1.0 = sous-espaces superposes. Verifier aussi la stabilite multi-seed.")


Matrice d'overlap (regime orthogonal, seed=0) :
  [1.000  0.000  0.000  0.000]
  [0.000  1.000  0.000  0.000]
  [0.000  0.000  1.000  0.000]
  [0.000  0.000  0.000  1.000]

Interpretation : diagonale = 1.0 (auto-overlap), hors-diagonale proche de 0 = orthogonalite,
proche de 1.0 = sous-espaces superposes. Verifier aussi la stabilite multi-seed.


## Limites et bilan honnete

- **H3 (sensibilite au bruit) INCONCLUSIVE par construction du generateur** : `synthesize_activations` utilise les memes bases orthogonales pour tous les niveaux de bruit ; le bruit est additif sur les activations, pas sur les directions factorielles elles-memes. L'overlap entre sous-espaces factoriels reste donc 0.0 par construction, independamment du bruit. Pour tester H3, il faudrait introduire le bruit **dans les bases** (rotation aleatoire des sous-espaces) -- un chantier a part.
- **H2 (additivite jointe) INCONCLUSIVE avec gap median 9 dim** : la dimension jointe (231) est legerement inferieure a la somme des dimensions factorielles (240). Le gap est petit (< 4% de la dimension jointe) et stable sur 5 seeds ; il reflete probablement la perte de variance au centrage pondere. Une investigation plus poussee (NC@80/90 vs NC@95) est en dehors du scope de ce grain.
- **Donnees synthetiques uniquement** : les primitives sont testees sur des generateurs connus, pas sur des activations reelles de transformer. La migration vers le contrat de trace v1 (#15476) ouvrira la voie aux activations SAE/J-Lens reelles.
- **Echelle** : N_DIM_RESIDUAL=256 et N_FACTORS=4 sont des proxies ; un vrai residual stream LLM est ~4096 dim avec un nombre de facteurs non connu a priori.
- **Ponderation** : les poids sont ici tous egaux (1/N). La primitive supporte une ponderation explicite ; les donnees reelles (futur #15476) pourront introduire des poids par token ou par etat belief.
- **Multi-seed 5 seeds** : conforme aux conventions notebook ICT-Series (>= 4 seeds pour probes).
- **Temoin nul (suivi du 15/09, LIVRE en deux temps)** : le null de H1/H4 reste un null
  de sous-espaces aleatoires apparies — la question « biais inductif de l'architecture »
  est maintenant couverte ailleurs : la boucle fermee entraine un modele reel sur le banc
  `ict/bench_factorise.py` avec le temoin d'initialisations non entrainees en
  percentiles 5/50/95 -- les trois percentiles sont imprimes par la boucle fermee
  elle-meme. La section des regimes generatifs (#20071)
  rejoue ce temoin a CHAQUE niveau de bruit du canal, mais elle n'en imprime que la
  **bande** p05-p95 (`[bande ...]`) : le triplet p05/p50/p95 n'y est pas rendu.
- **Axes de graine** : H4 separe data-seed (`[1000, seed]`) et null-seed (`[2000, seed]`)
  sur deux entropies disjointes. Depuis la boucle fermee, l'axe weight-init existe aussi
  (le temoin = initialisations non entrainees, le modele entraine etant a seed fixe) ;
  la matrice COMPLETE data-seed x weight-init-seed sur activations entrainees reste
  ouverte (suivi #15478).
- **Dependance d'environnement de la suite de tests** : `ict/tests/test_extracts_tiers.py`
  importe `cryptography` (via `ict/extracts_tiers.py`) ; sans ce paquet, la suite
  `ict/tests` echoue a la **collecte** — l'arret porte sur l'import du module de test,
  pas sur une assertion. Installation dans l'environnement d'execution :
  `pip install cryptography`.
- **Pas de code non licencie** : primitives reimplementees depuis arXiv:2602.02385 et nos propres contrats ; aucun copier-coller des depots `factored-reps`, `simplexity`, `strange-loop` ou `pytorch-AI-interpretability-transformer_ZM` (sans licence detectee au preflight #15475).
- **Verdict scientifique** : H1 SUPPORTED (5/5 seeds), H2 INCONCLUSIVE (gap 9 dim), H3 INCONCLUSIVE (generateur limite), H4 SUPPORTED (croisement a k_cross = 8 comme pre-enregistre). Le notebook delivre des primitives testees et un verdict falsifiable ; le passage aux activations reelles via #15476 est une etape distincte.


## Migration future vers le contrat de trace v1 (#15476)

Quand #15476 sera livre, le consommateur de ce notebook deviendra :

```python
from pathlib import Path
import numpy as np

trace_path = Path("ict/traces/v1/ict36_flens_orthogonal.npz")
data = np.load(trace_path)
activations = data["activations"]  # (N, D)
weights = data["weights"]          # (N,)
metadata = json.loads(data["metadata_json"].item() if hasattr(data["metadata_json"], "item") else data["metadata_json"])
mu, comps, sv, evr = weighted_pca(activations, weights)
```

Aucun changement aux primitives : le contrat ajoute provenance, semantique et validation sans modifier l'analyse.


## Sortie de reference (extrait verbatim)

Cette section est reservee a la sortie reelle du notebook apres execution. Les valeurs exactes dependent du kernel et du seed ; voir les cellules code ci-dessus pour les sorties verbatim. Le verdict final (cellules Verdict multi-seed et H4) est l'output scientifique livrable.
